# Petróleo vs. Bolsa Portuguesa (2000–2026): um Data Lakehouse com Apache Spark

**Unidade curricular:** Analyzing Big Data — EDSA
**Elementos:** Ana Filipa Filipe 20250313, José Justino 20251110, Inês Rodrigues 20251957, Sandra Figueiredo 20251705
**Plataforma:** Apache Spark em Databricks Free Edition (Serverless, Unity Catalog, Delta Lake)

---

### Porquê este tema
Portugal não produz petróleo e importa todo o crude que consome. Para a maior parte das empresas, um barril mais caro é um custo. Para a Galp, a única petrolífera do PSI, é receita. O projeto testa se a bolsa portuguesa mostra esta diferença.

### O que foi construído
Um lakehouse em Databricks que lê 21 séries de duas APIs públicas (FRED e Yahoo Finance) em três leituras. Cada leitura fica guardada em bruto e passa pelas camadas Bronze, Silver e Gold. As análises (queries SQL, gráficos e dois modelos de ML) leem só a Gold.

### Como está organizado
- Secções 0 a 5: o pipeline, da fonte até à Gold, com as leituras incrementais e o reprocessamento.
- Secção 6: as análises.
- Secção 7: as conclusões, que respondem às cinco perguntas abaixo.

### Objetivo
Construir um processo de dados **estruturado e incremental**, como num caso real, para estudar a relação entre o **preço do petróleo** (Brent e WTI) e a **bolsa portuguesa** (índice PSI e 15 empresas cotadas), de 2000 até hoje.

**Hipótese do grupo:** quando o preço do petróleo sobe, a cotação das empresas portuguesas desce — **exceto a Galp**. Para a Galp o petróleo é receita; para as outras empresas é um custo.

**Perguntas de análise**
1. Como evoluiu o preço do petróleo desde 2000, em dólares, em euros e em euros a preços de hoje (descontando a inflação)?
2. O mercado acionista português acompanha o petróleo? Em que períodos a relação é mais forte?
3. Como reage cada empresa ao petróleo, mês a mês? A Galp comporta-se de forma diferente das restantes? E depois de descontar o movimento geral da bolsa?
4. Como se comportaram os mercados nas grandes crises (2008, 2014-16, COVID-19, guerra na Ucrânia)?
5. Os movimentos do petróleo ajudam a explicar ou a prever os movimentos do PSI?

### Índice
- [Introdução: fontes, arquitetura e empresas](#intro)
- [0. Configuração do ambiente](#s0)
- [1. Ingestão → LANDING](#s1)
  - [1.1 Primeira leitura (carga histórica)](#s1-1)
- [2. Camada BRONZE](#s2)
  - [2.1 Qualidade dos dados em bruto](#s2-1)
- [3. Camada SILVER](#s3)
- [4. Camada GOLD](#s4)
  - [4.1 Gold das empresas](#s4-1)
- [5. Leituras incrementais e reprocessamento](#s5)
  - [5.1 Reprocessamento a partir da Bronze](#s5-1)
- [6. Análise](#s6)
  - [6.1 Queries](#s6-1)
  - [6.2 Dashboard e gráficos](#s6-2)
  - [6.3 Modelos de Machine Learning](#s6-3)
- [7. Conclusões](#s7)
  - [7.1 Sobre os dados](#s7-1) · [7.2 Sobre a hipótese](#s7-2) · [7.3 Sobre o processo](#s7-3) · [7.4 Sobre a tecnologia](#s7-4)

<a id="intro"></a>
### Fontes de dados (APIs públicas e gratuitas)

| Fonte | Série | Conteúdo | Frequência | Desde |
|---|---|---|---|---|
| **FRED** (Federal Reserve Bank of St. Louis) | `DCOILBRENTEU` | Brent, preço spot (USD/barril) | Diária | 1987 |
| FRED | `DCOILWTICO` | WTI, preço spot (USD/barril) | Diária | 1986 |
| FRED | `DEXUSEU` | Câmbio USD por 1 EUR | Diária | 1999 |
| FRED / OCDE | `SPASTT01PTM661N` | Índice de preços de ações — Portugal (2015 = 100) | Mensal | 1988 |
| FRED / Eurostat | `CP0000PTM086NEST` | IHPC Portugal — índice harmonizado de preços no consumidor (inflação) | Mensal | 1996 |
| **Yahoo Finance** (chart API) | `PSI20.LS` | Índice PSI (Euronext Lisbon): abertura, máximo, mínimo, fecho, volume | Diária | 2013 |
| Yahoo Finance | 15 empresas (`GALP.LS`, `EDP.LS`, …) | Cotações: abertura, máximo, mínimo, fecho, volume e **fecho ajustado** | Diária | 2000 ou entrada em bolsa |

Nota: não existe uma fonte gratuita com o PSI **diário** antes de 2013 (a Euronext só disponibiliza 2 anos). Por isso o período 2000–2013 é coberto pelo índice mensal da OCDE para Portugal, que acompanha o mercado acionista português.

Nas empresas usa-se o **fecho ajustado** (`adjclose`): em períodos longos, o fecho simples ignora os dividendos e as operações de capital (ex.: aumentos de capital do BCP) e distorce os retornos.

### Arquitetura (Medallion)

```
 APIs (FRED, Yahoo) ──► LANDING (ficheiros em bruto, 1 por leitura, no Volume)
                             │
                             ▼
                    BRONZE  (Delta, append-only: todo o histórico de leituras + metadados → permite reprocessar)
                             │  normalização: tipos, nulos, duplicados, datas, regras de qualidade
                             ▼
                    SILVER  (Delta, modelo detalhado por tema: dimensão instrumentos + factos diários/mensais, MERGE incremental)
                             │  integração, forward-fill, retornos, médias móveis, agregações
                             ▼
                    GOLD    (Delta, tabelas otimizadas para análise: diária, mensal, anual; particionadas + OPTIMIZE/Z-ORDER)
                             │
                             ▼
                    ANÁLISE (Spark SQL, gráficos/dashboard, modelos Spark MLlib)
```

### Empresas analisadas e hipótese

| Empresa | Símbolo | Grupo (setor) | Hipótese do grupo | Leitura por setor | Porquê |
|---|---|---|---|---|---|
| Galp Energia | `GALP.LS` | Petróleo e gás | **+** | + | O petróleo é receita |
| EDP | `EDP.LS` | Eletricidade e redes | − | ? | O preço da eletricidade acompanha o gás, que acompanha o petróleo |
| EDP Renováveis | `EDPR.LS` | Eletricidade e redes | − | ? | Idem |
| REN | `RENE.LS` | Eletricidade e redes | − | ? | Negócio regulado, pouco exposto |
| Navigator | `NVG.LS` | Indústria exportadora | − | ? | Gasta muita energia, mas vende em dólares e acompanha o ciclo das matérias-primas |
| Altri | `ALTR.LS` | Indústria exportadora | − | ? | Idem |
| Semapa | `SEM.LS` | Indústria exportadora | − | ? | Idem (dona da Navigator e da Secil) |
| Corticeira Amorim | `COR.LS` | Indústria exportadora | − | ? | Exportadora, custos de transporte |
| Jerónimo Martins | `JMT.LS` | Consumo e retalho | − | − | Combustível mais caro reduz o poder de compra |
| Sonae | `SON.LS` | Consumo e retalho | − | − | Idem |
| Ibersol | `IBS.LS` | Consumo e retalho | − | − | Idem |
| NOS | `NOS.LS` | Serviços domésticos | − | − | Custos de energia, procura interna |
| CTT | `CTT.LS` | Serviços domésticos | − | − | Custos de transporte |
| Mota-Engil | `EGL.LS` | Exposição a países petrolíferos | − | + | Negócio em Angola, Moçambique e América Latina, que ganham com o petróleo caro |
| BCP | `BCP.LS` | Exposição a países petrolíferos | − | + | Participação em Angola e Moçambique |

- A hipótese **testada** é a do grupo: sinal **+** para a Galp e **−** para todas as outras. A coluna "Leitura por setor" é uma expectativa mais fina, usada para interpretar os resultados.
- **Regra de método:** as correlações e os betas são calculados sobre **retornos mensais** (variações), nunca sobre níveis de preço, para evitar correlações espúrias (duas séries que sobem durante 20 anos parecem correlacionadas mesmo sem relação).
- **Enviesamento de sobrevivência:** analisam-se as 15 empresas que estão hoje no PSI. Empresas que saíram da bolsa (ex.: PT, BES, Cimpor, Brisa) não entram, o que deve ser tido em conta nas conclusões.

---
<a id="s0"></a>
## 0. Configuração do ambiente

Prepara o ambiente para o notebook correr de cima a baixo (*Run all*), no Databricks ou localmente: deteta onde está a correr, cria o schema e o Volume e define o catálogo das 21 séries.

**Preparação:** o notebook tenta ler os dados diretamente das APIs. O Databricks Free Edition só deixa aceder a domínios "confiáveis"; se uma API estiver bloqueada, o notebook usa os **ficheiros de *fallback*** (contingência), uma cópia antiga com a mesma estrutura da resposta da API. A fonte principal é sempre a API.
- A segunda célula desta secção cria o schema `workspace.oil_psi` e o Volume `lake` e copia os 21 ficheiros de *fallback* da pasta `dados_fallback/` (5 `fred_*.csv` e 16 `yahoo_*.json`) para `lake/fallback`. A cópia é idempotente: só copia o que falta ou mudou.
- *Só se o notebook foi importado sem a Git folder:* em **Catalog → workspace → oil_psi → Volumes → lake**, criar a pasta `fallback` e fazer *upload* dos 21 ficheiros de `dados_fallback/`.

In [ ]:
import os, json, datetime as dt, uuid, math
import requests
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import types as T

# Deteção do ambiente: no Databricks usamos Delta Lake + Unity Catalog.
# (O notebook também corre em Google Colab / Spark local, guardando as tabelas em Parquet.)
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    import sys, glob, subprocess
    # Os workers do Spark usam o mesmo Python do kernel (evita erros em Windows com vários Pythons instalados)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

    # O Spark precisa de Java 17 ou 21: se o JAVA_HOME não estiver definido, procuramos um JDK nos sítios habituais (macOS e Windows)
    if not os.environ.get("JAVA_HOME"):
        candidatos = []
        if sys.platform == "darwin":
            for v in ("17", "21"):
                r = subprocess.run(["/usr/libexec/java_home", "-v", v], capture_output=True, text=True)
                if r.returncode == 0:
                    candidatos.append(r.stdout.strip())
            for prefixo in ("/opt/homebrew/opt", "/usr/local/opt"):
                for v in ("17", "21"):
                    candidatos.append(f"{prefixo}/openjdk@{v}/libexec/openjdk.jdk/Contents/Home")
        elif os.name == "nt":
            for pasta in (r"C:\Program Files\Eclipse Adoptium", r"C:\Program Files\Java", r"C:\Program Files\Microsoft"):
                for v in ("17", "21"):
                    candidatos += sorted(glob.glob(os.path.join(pasta, f"jdk-{v}*")), reverse=True)
        java_home = next((c for c in candidatos if os.path.isdir(c)), None)
        if java_home:
            os.environ["JAVA_HOME"] = java_home
        else:
            print("AVISO: não encontrei o Java 17/21. Instale-o (ver README, 'Execução local') ou defina o JAVA_HOME.")
    print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

    # As tabelas locais (Parquet) ficam dentro da pasta do lakehouse, e não numa pasta spark-warehouse/ solta na raiz
    WAREHOUSE = os.path.abspath(os.path.join("lakehouse_oil_psi", "warehouse"))

    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.appName("oil_psi_lakehouse")
             .config("spark.sql.warehouse.dir", WAREHOUSE).getOrCreate())
    def display(df, n=20):
        print(df.limit(n).toPandas().to_string())

    # As células %sql são do Databricks. Localmente, convertemos "%sql" em "%%sql" e corremos com spark.sql + display
    try:
        ip = get_ipython()
        def _sql_magic(line, cell):
            display(spark.sql(cell))
        ip.register_magic_function(_sql_magic, magic_kind="cell", magic_name="sql")
        def _sql_para_magic(lines):
            if lines and lines[0].strip() == "%sql":
                return ["%%sql\n"] + lines[1:]
            return lines
        ip.input_transformers_cleanup[:] = [t for t in ip.input_transformers_cleanup
                                            if getattr(t, "__name__", "") != "_sql_para_magic"] + [_sql_para_magic]
    except NameError:
        pass   # fora do IPython/Jupyter (ex.: script Python): as células %sql não se aplicam

TABLE_FORMAT = "delta" if IS_DATABRICKS else "parquet"

CATALOG, SCHEMA, VOLUME = "workspace", "oil_psi", "lake"
BASE    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}" if IS_DATABRICKS else os.path.abspath("lakehouse_oil_psi")
LANDING = f"{BASE}/landing"      # camada de aterragem: ficheiros em bruto exatamente como vieram da API
FALLBACK = f"{BASE}/fallback"   # contingência: cópia antiga das APIs, só usada quando a API falha

RESET = True    # True = apagar tudo e reconstruir de raiz (demonstração). Em produção ficaria False.

print("Databricks:", IS_DATABRICKS, "| formato das tabelas:", TABLE_FORMAT)
print("Base do lakehouse:", BASE)

In [ ]:
# Criar o schema (base de dados) e o Volume que guarda os ficheiros do lakehouse
if IS_DATABRICKS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA} COMMENT 'Lakehouse: petróleo vs PSI'")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME} COMMENT 'Ficheiros landing e fallback'")
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
else:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA}")
    spark.sql(f"USE {SCHEMA}")

ALL_TABLES = ["ctl_ingestion_log", "bz_fred_observations", "bz_yahoo_chart",
              "sv_dim_instrument", "sv_fact_price_daily", "sv_fact_equity_daily", "sv_fact_pt_shares_monthly", "sv_fact_cpi_monthly",
              "gd_market_daily", "gd_market_monthly", "gd_market_yearly", "gd_equity_monthly", "gd_oil_sensitivity",
              "sv_fact_psi_daily"]   # sv_fact_psi_daily: tabela da versão anterior (o PSI passou para sv_fact_equity_daily)

# Localmente o catálogo é em memória: ao reiniciar o kernel esquece as tabelas, mas os ficheiros Parquet ficam no disco
# e o DROP TABLE não os apaga (o saveAsTable falharia com LOCATION_ALREADY_EXISTS). Por isso o RESET local apaga também as pastas.
import shutil
pasta_tabela = lambda t: os.path.join(WAREHOUSE, f"{SCHEMA}.db", t)
if RESET:
    for t in ALL_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        if not IS_DATABRICKS:
            shutil.rmtree(pasta_tabela(t), ignore_errors=True)
    shutil.rmtree(LANDING, ignore_errors=True)   # o fallback NÃO é apagado
elif not IS_DATABRICKS:
    orfas = [t for t in ALL_TABLES if os.path.isdir(pasta_tabela(t)) and not spark.catalog.tableExists(t)]
    if orfas:
        print("AVISO: com RESET = False, as tabelas de um kernel anterior não existem no catálogo local (é em memória):",
              ", ".join(orfas), "\n        A escrita vai falhar com LOCATION_ALREADY_EXISTS. Localmente, use RESET = True.")
os.makedirs(LANDING, exist_ok=True)

# Ficheiros de fallback: copiados automaticamente da pasta dados_fallback/ do repositório (localmente ou numa Git folder do Databricks,
# onde a pasta de trabalho é a do notebook). Idempotente: só copia o que falta ou mudou; se não houver dados_fallback/, faz-se o upload manual.
import shutil
DADOS_FALLBACK = next((p for p in (os.path.abspath("dados_fallback"), os.path.abspath(os.path.join("..", "dados_fallback")))
                    if os.path.isdir(p)), None)
if DADOS_FALLBACK:
    os.makedirs(FALLBACK, exist_ok=True)
    copiados, existentes = 0, 0
    for f in sorted(os.listdir(DADOS_FALLBACK)):
        origem, destino = os.path.join(DADOS_FALLBACK, f), os.path.join(FALLBACK, f)
        if not os.path.isfile(origem) or f.startswith("."):
            continue
        if os.path.isfile(destino) and os.path.getsize(destino) == os.path.getsize(origem):
            existentes += 1
        else:
            shutil.copyfile(origem, destino)
            copiados += 1
    print(f"Fallback de {DADOS_FALLBACK}: {copiados} copiados, {existentes} já existiam em {FALLBACK}")
else:
    print("Pasta dados_fallback/ não encontrada: os ficheiros de fallback têm de ser carregados à mão para", FALLBACK)

print("Ficheiros de fallback disponíveis:", sorted(os.listdir(FALLBACK)) if os.path.isdir(FALLBACK) else "nenhum (só será possível ler das APIs)")

In [ ]:
# Catálogo das séries que alimentam o lakehouse (metadados de negócio)
FRED_SERIES = {
    "DCOILBRENTEU":    {"name": "Brent - preço spot",                         "unit": "USD/barril",   "currency": "USD", "freq": "D", "category": "Petróleo"},
    "DCOILWTICO":      {"name": "WTI - preço spot",                           "unit": "USD/barril",   "currency": "USD", "freq": "D", "category": "Petróleo"},
    "DEXUSEU":         {"name": "Câmbio EUR/USD (USD por 1 EUR)",             "unit": "USD por EUR",  "currency": "USD", "freq": "D", "category": "Câmbio"},
    "SPASTT01PTM661N": {"name": "OCDE - índice de preços de ações Portugal",   "unit": "Índice 2015=100", "currency": "EUR", "freq": "M", "category": "Ações PT"},
    "CP0000PTM086NEST": {"name": "IHPC Portugal - índice harmonizado de preços no consumidor", "unit": "Índice", "currency": "EUR", "freq": "M", "category": "Inflação"},
}

# Empresas: sector = grupo de setor; hypothesis = sinal esperado pela hipótese do grupo (petróleo sobe → ação desce,
# exceto a Galp); sector_view = sinal esperado pela leitura do setor ('?' = ambíguo)
def _eq(name, sector, hypothesis, sector_view):
    return {"name": name, "unit": "EUR", "currency": "EUR", "freq": "D", "category": "Ações PT",
            "sector": sector, "hypothesis": hypothesis, "sector_view": sector_view}
YAHOO_SYMBOLS = {
    "PSI20.LS":        {"name": "PSI - Euronext Lisbon",                     "unit": "Pontos",       "currency": "EUR", "freq": "D", "category": "Ações PT"},
    "GALP.LS": _eq("Galp Energia",      "Petróleo e gás",                  "+", "+"),
    "EDP.LS":  _eq("EDP",               "Eletricidade e redes",            "-", "?"),
    "EDPR.LS": _eq("EDP Renováveis",    "Eletricidade e redes",            "-", "?"),
    "RENE.LS": _eq("REN",               "Eletricidade e redes",            "-", "?"),
    "NVG.LS":  _eq("Navigator",         "Indústria exportadora",           "-", "?"),
    "ALTR.LS": _eq("Altri",             "Indústria exportadora",           "-", "?"),
    "SEM.LS":  _eq("Semapa",            "Indústria exportadora",           "-", "?"),
    "COR.LS":  _eq("Corticeira Amorim", "Indústria exportadora",           "-", "?"),
    "JMT.LS":  _eq("Jerónimo Martins",  "Consumo e retalho",               "-", "-"),
    "SON.LS":  _eq("Sonae",             "Consumo e retalho",               "-", "-"),
    "IBS.LS":  _eq("Ibersol",           "Consumo e retalho",               "-", "-"),
    "NOS.LS":  _eq("NOS",               "Serviços domésticos",             "-", "-"),
    "CTT.LS":  _eq("CTT",               "Serviços domésticos",             "-", "-"),
    "EGL.LS":  _eq("Mota-Engil",        "Exposição a países petrolíferos", "-", "+"),
    "BCP.LS":  _eq("BCP",               "Exposição a países petrolíferos", "-", "+"),
}

---
<a id="s1"></a>
## 1. Ingestão: ler os dados da fonte → camada LANDING

O primeiro passo traz os dados da fonte para o lakehouse sem os alterar. Se uma regra mais à frente estiver errada, o original continua guardado e pode ser processado de novo.

Cada leitura (*batch*) chama a API com uma janela de datas `[start, end]` e grava a resposta **sem qualquer alteração** num ficheiro novo:

`landing/<fonte>/<série>/ingest_date=AAAA-MM-DD/<série>_<batch_id>.csv|json`

Assim cada leitura fica guardada para sempre e pode ser reprocessada. Cada leitura também é registada na tabela de controlo `ctl_ingestion_log`.

In [ ]:
# O User-Agent depende da fonte (testado em 11-11-2026): o Yahoo recusa o do requests (HTTP 429) e precisa de um tipo "browser";
# o FRED faz o contrário, não responde a User-Agents tipo "browser" (timeout) e responde logo ao do requests. Não unificar.
YAHOO_HEADERS = {"User-Agent": "Mozilla/5.0 (EDSA ABD student project)"}

def _http_get(url, params, headers=None):
    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text

def _to_epoch(d):
    return int(dt.datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=dt.timezone.utc).timestamp())

def fetch_fred(series_id, start, end):
    """Lê uma série do FRED (CSV). Se a API não estiver acessível, usa o ficheiro de fallback com a mesma janela de datas."""
    try:
        txt = _http_get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                        {"id": series_id, "cosd": start, "coed": end})
        if not txt.startswith("observation_date"):
            raise ValueError("resposta inesperada da API")
        return txt, "API"
    except Exception as e:
        lines = open(find_fallback(f"fred_{series_id}.csv"), encoding="utf-8").read().strip().splitlines()
        body = [l for l in lines[1:] if start <= l[:10] <= end]
        return "\n".join([lines[0]] + body) + "\n", f"FALLBACK ({type(e).__name__})"

def fetch_yahoo(symbol, start, end):
    """Lê o histórico diário de um índice do Yahoo Finance (JSON). Contingência: ficheiro de fallback filtrado pela janela."""
    p1, p2 = _to_epoch(start), _to_epoch(end) + 86400
    try:
        txt = _http_get(f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
                        {"period1": p1, "period2": p2, "interval": "1d"}, YAHOO_HEADERS)
        json.loads(txt)["chart"]["result"][0]["timestamp"]
        return txt, "API"
    except Exception as e:
        j = json.load(open(find_fallback(f"yahoo_{symbol}.json"), encoding="utf-8"))
        res = j["chart"]["result"][0]
        idx = [i for i, t in enumerate(res["timestamp"]) if p1 <= t < p2]
        res["timestamp"] = [res["timestamp"][i] for i in idx]
        for block in ("quote", "adjclose"):
            for serie in res["indicators"].get(block, []):
                for k in serie:
                    serie[k] = [serie[k][i] for i in idx]
        return json.dumps(j), f"FALLBACK ({type(e).__name__})"

def find_fallback(filename):
    """Procura o ficheiro de fallback na pasta fallback/ e, se não estiver lá, em qualquer pasta do Volume (ignora maiúsculas)."""
    for root, _, files in os.walk(BASE):
        if "/landing" in root:
            continue
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    raise FileNotFoundError(f"A API está bloqueada e não encontrei '{filename}' no Volume {BASE}. "
                            f"Faça upload dos ficheiros da pasta dados_fallback para {FALLBACK}/")

def land(source, series, payload, ext, batch_id, ingest_date):
    folder = f"{LANDING}/{source}/{series}/ingest_date={ingest_date}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{series}_{batch_id}.{ext}"
    with open(path, "w", encoding="utf-8") as f:
        f.write(payload)
    return path

In [ ]:
from concurrent.futures import ThreadPoolExecutor

LOG_SCHEMA = T.StructType([
    T.StructField("batch_id", T.StringType()), T.StructField("batch_name", T.StringType()),
    T.StructField("source", T.StringType()), T.StructField("series_id", T.StringType()),
    T.StructField("window_start", T.StringType()), T.StructField("window_end", T.StringType()),
    T.StructField("extract_mode", T.StringType()), T.StructField("records", T.LongType()),
    T.StructField("landing_file", T.StringType()), T.StructField("ingestion_ts", T.TimestampType()),
])

def ingest(batch_name, windows, ingest_date=None):
    """Executa uma leitura (batch) de todas as séries. windows = {series_id: (start, end)}.
    Devolve o batch_id e a lista de ficheiros gravados na landing."""
    batch_id = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    ingest_date = ingest_date or dt.date.today().isoformat()
    now = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

    def read_one(source, sid):
        start, end = windows[sid]
        if source == "FRED":
            txt, mode = fetch_fred(sid, start, end)
            path = land("fred", sid, txt, "csv", batch_id, ingest_date)
            n = max(len(txt.strip().splitlines()) - 1, 0)
        else:
            txt, mode = fetch_yahoo(sid, start, end)
            path = land("yahoo", sid, txt, "json", batch_id, ingest_date)
            n = len(json.loads(txt)["chart"]["result"][0].get("timestamp") or [])
        return (batch_id, batch_name, source, sid, start, end, mode, n, path, now)

    # Os pedidos às APIs são feitos em paralelo (8 de cada vez): o tempo é quase todo espera pela rede, não CPU.
    # Cada série continua com o seu próprio fallback; o map devolve os resultados pela ordem do catálogo.
    pedidos = [("FRED", s) for s in FRED_SERIES] + [("YAHOO", s) for s in YAHOO_SYMBOLS]
    with ThreadPoolExecutor(max_workers=8) as ex:
        log = list(ex.map(lambda p: read_one(*p), pedidos))
    files = [(r[2].lower(), r[8]) for r in log]
    spark.createDataFrame(log, LOG_SCHEMA).write.format(TABLE_FORMAT).mode("append").saveAsTable("ctl_ingestion_log")
    for r in log:
        print(f"  {r[2]:5} {r[3]:16} {r[4]} → {r[5]}  {r[7]:>6} registos  [{r[6]}]")
    return batch_id, files

<a id="s1-1"></a>
### 1.1 Primeira leitura — carga histórica (2000-01-01 a 2025-12-31)
Para simular um processo real, a carga é feita em **três leituras**: a carga histórica inicial agora e duas leituras incrementais na secção 5 (até 30-06-2026 e até hoje).

In [ ]:
# Diagnóstico: listar os ficheiros do Volume (exceto a landing) para confirmar que os ficheiros de fallback estão em fallback/
for root, dirs, files in os.walk(BASE):
    if "/landing" not in root:
        print(root, files)

In [ ]:
HIST_START, HIST_END = "2000-01-01", "2025-12-31"
all_series = list(FRED_SERIES) + list(YAHOO_SYMBOLS)

batch1_id, batch1_files = ingest("carga_inicial", {s: (HIST_START, HIST_END) for s in all_series})

In [ ]:
# Verificar os ficheiros em bruto na landing (exatamente como vieram da fonte)
for src, path in batch1_files:
    print(path.replace(BASE, "…"), f"{os.path.getsize(path)/1024:.0f} KB")
print()
print(open(batch1_files[0][1]).read()[:200])

**Fim da secção 1.** A primeira leitura deixou na landing um ficheiro por série, tal como veio da API, e o registo de cada leitura no `ctl_ingestion_log`. A secção seguinte carrega estes ficheiros para tabelas.

---
<a id="s2"></a>
## 2. Camada BRONZE — histórico de leituras (pronto para reprocessamento)

Os ficheiros em bruto passam a tabelas, sem limpeza nem deduplicação. A Bronze é o único sítio onde ficam todas as leituras, por isso é a partir dela que a Silver se reconstrói na secção 5.1.

A Bronze guarda **todas as leituras**, sem descartar nada (*append-only*), mantendo os valores tal como vieram (texto/número em bruto) e acrescentando metadados de linhagem:
`_batch_id`, `_ingestion_ts`, `_source_file`, `_ingest_date`.

- `bz_fred_observations` — uma linha por (série, data, leitura)
- `bz_yahoo_chart` — uma linha por (símbolo, timestamp, leitura), obtida a partir do JSON aninhado da API

In [ ]:
def load_bronze_fred(paths, batch_id):
    dfs = []
    for p in paths:
        series_id = p.split("/fred/")[1].split("/")[0]
        raw = spark.read.option("header", True).csv(p)                    # tudo como string (dados em bruto)
        dfs.append(raw.select(
            F.lit(series_id).alias("series_id"),
            F.col("observation_date").alias("observation_date"),
            F.col(series_id).alias("value_raw"),
            F.lit(batch_id).alias("_batch_id"),
            F.current_timestamp().alias("_ingestion_ts"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.regexp_extract(F.col("_metadata.file_path"), r"ingest_date=([0-9-]+)", 1).alias("_ingest_date"),
        ))
    out = dfs[0]
    for d in dfs[1:]:
        out = out.unionByName(d)
    out.write.format(TABLE_FORMAT).mode("append").partitionBy("_ingest_date").saveAsTable("bz_fred_observations")
    return out.count()

def load_bronze_yahoo(paths, batch_id):
    raw = spark.read.option("multiLine", True).json(paths)
    res = raw.select(F.explode("chart.result").alias("r"), F.col("_metadata.file_path").alias("_source_file"))
    q = "r.indicators.quote"
    # fecho ajustado a dividendos e operações de capital (bloco indicators.adjclose do JSON); se a API não o devolver, fica nulo
    has_adj = "adjclose" in res.schema["r"].dataType["indicators"].dataType.fieldNames()
    adj_col = F.col("r.indicators.adjclose.adjclose")[0][F.col("i")].cast("double") if has_adj else F.lit(None).cast("double")
    out = (res.select("*", F.posexplode("r.timestamp").alias("i", "ts_epoch"))
              .select(
                  F.col("r.meta.symbol").alias("symbol"),
                  F.col("r.meta.currency").alias("currency"),
                  F.col("r.meta.exchangeTimezoneName").alias("exchange_tz"),
                  F.col("ts_epoch").cast("long").alias("ts_epoch"),
                  F.col(f"{q}.open")[0][F.col("i")].alias("open_raw"),
                  F.col(f"{q}.high")[0][F.col("i")].alias("high_raw"),
                  F.col(f"{q}.low")[0][F.col("i")].alias("low_raw"),
                  F.col(f"{q}.close")[0][F.col("i")].alias("close_raw"),
                  F.col(f"{q}.volume")[0][F.col("i")].alias("volume_raw"),
                  adj_col.alias("adjclose_raw"),
                  F.lit(batch_id).alias("_batch_id"),
                  F.current_timestamp().alias("_ingestion_ts"),
                  F.col("_source_file"),
                  F.regexp_extract("_source_file", r"ingest_date=([0-9-]+)", 1).alias("_ingest_date"),
              ))
    out.write.format(TABLE_FORMAT).mode("append").partitionBy("_ingest_date").saveAsTable("bz_yahoo_chart")
    return out.count()

def load_bronze(batch_id, files):
    n1 = load_bronze_fred([p for s, p in files if s == "fred"], batch_id)
    n2 = load_bronze_yahoo([p for s, p in files if s == "yahoo"], batch_id)
    print(f"Bronze ← batch {batch_id}: {n1} linhas FRED, {n2} linhas Yahoo")

load_bronze(batch1_id, batch1_files)

In [ ]:
display(spark.table("bz_fred_observations").limit(5))
display(spark.table("bz_yahoo_chart").limit(5))
spark.table("bz_yahoo_chart").printSchema()

<a id="s2-1"></a>
### 2.1 Qualidade dos dados em bruto
Antes de normalizar, medimos os problemas que existem na fonte: valores em falta (feriados, dias sem cotação), duplicados e tipos de dados (no FRED tudo chega como texto).

In [ ]:
bz_f = spark.table("bz_fred_observations")
display(bz_f.groupBy("series_id").agg(
    F.count("*").alias("linhas"),
    F.sum(F.when((F.col("value_raw").isNull()) | (F.trim("value_raw").isin("", ".")), 1).otherwise(0)).alias("valores_em_falta"),
    F.min("observation_date").alias("primeira_data"), F.max("observation_date").alias("ultima_data"),
    (F.count("*") - F.countDistinct("observation_date")).alias("duplicados"),
).orderBy("series_id"))

bz_y = spark.table("bz_yahoo_chart")
display(bz_y.groupBy("symbol").agg(
    F.count("*").alias("linhas"),
    F.sum(F.when(F.col("close_raw").isNull(), 1).otherwise(0)).alias("fecho_em_falta"),
    F.sum(F.when(F.col("adjclose_raw").isNull(), 1).otherwise(0)).alias("ajustado_em_falta"),
    F.sum(F.when(F.col("volume_raw") == 0, 1).otherwise(0)).alias("volume_zero"),
    F.from_unixtime(F.min("ts_epoch")).alias("primeiro"), F.from_unixtime(F.max("ts_epoch")).alias("ultimo"),
).orderBy("symbol"))

**Fim da secção 2.** A Bronze tem as duas tabelas, com os metadados de linhagem, e os problemas da fonte estão medidos: valores em falta, duplicados e, no FRED, tudo em texto. A Silver trata cada um com uma regra explícita.

---
<a id="s3"></a>
## 3. Camada SILVER — dados normalizados e modelados para o tema

Aqui os dados ficam corretos e organizados para o tema: uma dimensão de instrumentos e factos diários e mensais. Cada problema medido em 2.1 tem uma regra que o resolve.

**Regras de normalização ("data rules")**
1. Tipos: `value_raw` (texto) → `double`; datas em texto → `date`; *timestamps* Unix → data local de Lisboa.
2. Nulos: valores vazios ou `.` (convenção do FRED para "sem observação") são eliminados; no PSI, dias sem fecho são eliminados.
3. Duplicados: a mesma (série, data) pode chegar em várias leituras. Fica sempre a **leitura mais recente** (`_ingestion_ts`), o que também corrige revisões da fonte.
4. Validação: preços ≤ 0 ficam marcados com `dq_flag` em vez de serem apagados (ex.: o WTI fechou a **–37,63 USD** em 20-04-2020, um valor real).
5. Ações: guarda-se o fecho e o **fecho ajustado** (`adj_close`, a dividendos e operações de capital); os retornos das empresas são sempre calculados com o fecho ajustado. Os preços ficam com 4 casas decimais (há ações abaixo de 1 €, como o BCP).

**Modelo Silver (detalhe máximo, orientado ao tema)**
- `sv_dim_instrument` — dimensão com os instrumentos (série, nome, unidade, moeda, frequência, categoria, setor, sinal esperado pela hipótese, fonte)
- `sv_fact_price_daily` — factos diários do petróleo e do câmbio (data, instrumento, valor)
- `sv_fact_equity_daily` — factos diários do PSI e das 15 empresas (OHLC, volume e fecho ajustado)
- `sv_fact_pt_shares_monthly` — facto mensal do índice OCDE de ações portuguesas
- `sv_fact_cpi_monthly` — facto mensal do IHPC de Portugal (inflação)

A escrita é **incremental** com `MERGE` (*upsert*) pelas chaves de negócio.

In [ ]:
def upsert(df, table, keys):
    """Upsert incremental: MERGE INTO no Delta Lake (ou substituição equivalente em Parquet fora do Databricks)."""
    if not spark.catalog.tableExists(table):
        df.write.format(TABLE_FORMAT).saveAsTable(table)
        return
    if TABLE_FORMAT == "delta":
        df.createOrReplaceTempView("_src")
        cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
        spark.sql(f"""MERGE INTO {table} t USING _src s ON {cond}
                      WHEN MATCHED THEN UPDATE SET *
                      WHEN NOT MATCHED THEN INSERT *""")
    else:
        merged = spark.table(table).join(df.select(*keys), keys, "left_anti").unionByName(df).localCheckpoint()
        merged.write.format(TABLE_FORMAT).mode("overwrite").saveAsTable(table)

def latest_per_key(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.col("_ingestion_ts").desc(), F.col("_batch_id").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [ ]:
# Dimensão de instrumentos (dados de referência), com o setor e o sinal esperado pela hipótese do grupo
def _dim_row(k, v, source):
    return (k, v["name"], v["unit"], v["currency"], v["freq"], v["category"],
            v.get("sector"), v.get("hypothesis"), v.get("sector_view"), source)
dim_rows = [_dim_row(k, v, "FRED") for k, v in FRED_SERIES.items()] + \
           [_dim_row(k, v, "Yahoo Finance") for k, v in YAHOO_SYMBOLS.items()]
dim_instrument = spark.createDataFrame(dim_rows, "instrument_id string, name string, unit string, currency string, frequency string, "
                                                 "category string, sector string, oil_hypothesis string, sector_view string, source string")
upsert(dim_instrument, "sv_dim_instrument", ["instrument_id"])
display(spark.table("sv_dim_instrument"))

In [ ]:
def normalize_fred(bz):
    df = (bz.withColumn("value", F.when(F.trim("value_raw").isin("", "."), None).otherwise(F.col("value_raw")).cast("double"))
            .withColumn("date", F.to_date("observation_date", "yyyy-MM-dd"))
            .filter(F.col("value").isNotNull() & F.col("date").isNotNull()))
    df = latest_per_key(df, ["series_id", "date"])
    return df.select(
        F.col("series_id").alias("instrument_id"), "date", "value",
        F.when(F.col("value") <= 0, "PRECO_NAO_POSITIVO").alias("dq_flag"),
        "_batch_id", F.col("_ingestion_ts").alias("_source_ingestion_ts"), F.current_timestamp().alias("_updated_ts"))

def normalize_yahoo(bz):
    df = (bz.withColumn("date", F.to_date(F.from_utc_timestamp(F.timestamp_seconds("ts_epoch"), "Europe/Lisbon")))
            .filter(F.col("close_raw").isNotNull()))
    df = latest_per_key(df, ["symbol", "date"])
    return df.select(
        F.col("symbol").alias("instrument_id"), "date",
        F.round(F.col("open_raw").cast("double"), 4).alias("open"),
        F.round(F.col("high_raw").cast("double"), 4).alias("high"),
        F.round(F.col("low_raw").cast("double"), 4).alias("low"),
        F.round(F.col("close_raw").cast("double"), 4).alias("close"),
        # fecho ajustado a dividendos e operações de capital; se faltar, usa-se o fecho simples
        F.round(F.coalesce(F.col("adjclose_raw"), F.col("close_raw")).cast("double"), 6).alias("adj_close"),
        F.col("volume_raw").cast("long").alias("volume"),
        F.when(F.col("close_raw") <= 0, "PRECO_NAO_POSITIVO")
         .when(F.col("high_raw") < F.col("low_raw"), "MAX_MENOR_QUE_MIN").alias("dq_flag"),
        "_batch_id", F.col("_ingestion_ts").alias("_source_ingestion_ts"), F.current_timestamp().alias("_updated_ts"))

SILVER_FACTS = ["sv_fact_price_daily", "sv_fact_equity_daily", "sv_fact_pt_shares_monthly", "sv_fact_cpi_monthly"]

def build_silver(batch_id=None):
    """Normaliza a Bronze e faz MERGE na Silver. batch_id=None → reprocessa TODO o histórico da Bronze."""
    bz_f = spark.table("bz_fred_observations")
    bz_y = spark.table("bz_yahoo_chart")
    if batch_id:
        bz_f, bz_y = bz_f.filter(F.col("_batch_id") == batch_id), bz_y.filter(F.col("_batch_id") == batch_id)
    fred = normalize_fred(bz_f)
    daily_ids  = [k for k, v in FRED_SERIES.items() if v["freq"] == "D"]
    shares_ids = [k for k, v in FRED_SERIES.items() if v["freq"] == "M" and v["category"] == "Ações PT"]
    cpi_ids    = [k for k, v in FRED_SERIES.items() if v["freq"] == "M" and v["category"] == "Inflação"]
    upsert(fred.filter(F.col("instrument_id").isin(daily_ids)), "sv_fact_price_daily", ["instrument_id", "date"])
    upsert(fred.filter(F.col("instrument_id").isin(shares_ids)).withColumnRenamed("date", "month_date"),
           "sv_fact_pt_shares_monthly", ["instrument_id", "month_date"])
    upsert(fred.filter(F.col("instrument_id").isin(cpi_ids)).withColumnRenamed("date", "month_date"),
           "sv_fact_cpi_monthly", ["instrument_id", "month_date"])
    upsert(normalize_yahoo(bz_y), "sv_fact_equity_daily", ["instrument_id", "date"])   # PSI + 15 empresas
    for t in SILVER_FACTS:
        print(f"{t:28} {spark.table(t).count():>7} linhas")

build_silver(batch1_id)

In [ ]:
display(spark.table("sv_fact_price_daily").groupBy("instrument_id").agg(
    F.count("*").alias("dias"), F.min("date").alias("de"), F.max("date").alias("ate"),
    F.round(F.min("value"), 2).alias("min"), F.round(F.max("value"), 2).alias("max"),
    F.sum(F.when(F.col("dq_flag").isNotNull(), 1).otherwise(0)).alias("alertas_qualidade")))
display(spark.table("sv_fact_price_daily").filter("dq_flag IS NOT NULL"))
display(spark.table("sv_fact_equity_daily").groupBy("instrument_id").agg(
    F.count("*").alias("dias"), F.min("date").alias("de"), F.max("date").alias("ate"),
    F.round(F.min("adj_close"), 4).alias("min_ajustado"), F.round(F.max("adj_close"), 4).alias("max_ajustado"),
    F.sum(F.when(F.col("dq_flag").isNotNull(), 1).otherwise(0)).alias("alertas_qualidade")).orderBy("instrument_id"))

**Fim da secção 3.** Cada série tem um registo por data, com o tipo certo. Os preços ≤ 0, como o WTI de 20-04-2020, ficam marcados com `dq_flag` em vez de apagados. A Silver tem uma linha por instrumento e data; a Gold junta tudo numa tabela por data.

---
<a id="s4"></a>
## 4. Camada GOLD — dados preparados e otimizados para análise

A Gold é a camada que as análises leem. Junta as séries num calendário comum, converte o petróleo para euros e guarda retornos, médias e correlações já calculados, para que as queries não precisem de *joins*.

**Preparação**
- Integração das séries numa **calendário único** de dias úteis (união das datas com cotação).
- *Forward-fill* dos preços em feriados de um dos mercados (ex.: Brent em feriados do Reino Unido, PSI em feriados portugueses), com indicadores `*_quoted` para saber se o valor foi observado.
- **Brent em euros** = Brent USD ÷ EUR/USD (o PSI está em euros, por isso esta é a comparação justa).
- **Inflação:** o IHPC de Portugal permite exprimir o Brent em **euros a preços de hoje** (`brent_eur_real`), usado nos gráficos de evolução. Nas correlações usam-se retornos nominais: a inflação mensal (~0,2%) afeta por igual as duas séries e quase não muda os resultados.
- Retornos logarítmicos, médias móveis de 50 e 200 dias, correlação móvel de 1 ano (252 dias úteis), máximos e *drawdowns*.

**Otimização**
- Tabelas largas (*denormalizadas*), sem *joins* na hora da consulta.
- `gd_market_daily` **particionada por ano** + `OPTIMIZE … ZORDER BY (date)` (compactação de ficheiros pequenos e *data skipping*).
- Agregados pré-calculados: `gd_market_monthly` (2000–2026) e `gd_market_yearly`.
- Tabelas das empresas (secção 4.1): `gd_equity_monthly` e `gd_oil_sensitivity`.

In [ ]:
def build_gold():
    px = spark.table("sv_fact_price_daily").groupBy("date").pivot("instrument_id", ["DCOILBRENTEU", "DCOILWTICO", "DEXUSEU"]).agg(F.first("value"))
    px = px.withColumnRenamed("DCOILBRENTEU", "brent_usd_obs").withColumnRenamed("DCOILWTICO", "wti_usd_obs").withColumnRenamed("DEXUSEU", "eurusd_obs")
    psi = (spark.table("sv_fact_equity_daily").filter(F.col("instrument_id") == "PSI20.LS")
              .select("date", F.col("close").alias("psi_close_obs"), F.col("volume").alias("psi_volume")))
    spine = px.select("date").union(psi.select("date")).distinct().filter(F.dayofweek("date").between(2, 6))
    d = spine.join(px, "date", "left").join(psi, "date", "left")

    w_ff = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)    # série única e pequena → janela global
    for c in ["brent_usd", "wti_usd", "eurusd", "psi_close"]:
        d = d.withColumn(c, F.last(f"{c}_obs", ignorenulls=True).over(w_ff))
    d = (d.withColumn("brent_quoted", F.col("brent_usd_obs").isNotNull())
          .withColumn("psi_traded", F.col("psi_close_obs").isNotNull())
          .drop("brent_usd_obs", "wti_usd_obs", "eurusd_obs", "psi_close_obs")
          .withColumn("brent_eur", F.round(F.col("brent_usd") / F.col("eurusd"), 4)))

    w = Window.orderBy("date")
    for c, r in [("brent_usd", "ret_brent_usd"), ("brent_eur", "ret_brent_eur"), ("eurusd", "ret_eurusd"), ("psi_close", "ret_psi")]:
        d = d.withColumn(r, F.log(F.col(c) / F.lag(c).over(w)))
    w50, w200, w252 = [Window.orderBy("date").rowsBetween(-(n - 1), 0) for n in (50, 200, 252)]
    w_all = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
    d = (d.withColumn("brent_eur_ma50", F.avg("brent_eur").over(w50))
          .withColumn("brent_eur_ma200", F.avg("brent_eur").over(w200))
          .withColumn("psi_ma50", F.avg("psi_close").over(w50))
          .withColumn("psi_ma200", F.avg("psi_close").over(w200))
          .withColumn("corr_1y_psi_brent", F.when(F.count("ret_psi").over(w252) >= 200, F.corr("ret_psi", "ret_brent_eur").over(w252)))
          .withColumn("vol_1y_psi", F.stddev("ret_psi").over(w252) * math.sqrt(252))
          .withColumn("vol_1y_brent", F.stddev("ret_brent_eur").over(w252) * math.sqrt(252))
          .withColumn("psi_drawdown", F.col("psi_close") / F.max("psi_close").over(w_all) - 1)
          .withColumn("brent_eur_drawdown", F.col("brent_eur") / F.max("brent_eur").over(w_all) - 1)
          .withColumn("year", F.year("date")).withColumn("quarter", F.quarter("date")).withColumn("month", F.month("date")))
    (d.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true")
       .partitionBy("year").saveAsTable("gd_market_daily"))

    # ---- Mensal (2000+): petróleo + índice OCDE de ações PT + PSI (desde 2013)
    daily = spark.table("gd_market_daily")
    m = (daily.withColumn("month_date", F.trunc("date", "month")).groupBy("month_date").agg(
            F.round(F.avg("brent_usd"), 2).alias("brent_usd_avg"),
            F.round(F.avg("brent_eur"), 2).alias("brent_eur_avg"),
            F.round(F.avg("wti_usd"), 2).alias("wti_usd_avg"),
            F.round(F.avg("eurusd"), 4).alias("eurusd_avg"),
            F.round(F.max_by("brent_eur", "date"), 2).alias("brent_eur_eom"),
            F.round(F.max_by("psi_close", "date"), 2).alias("psi_eom"),
            F.round(F.avg("psi_close"), 2).alias("psi_avg")))
    oecd = spark.table("sv_fact_pt_shares_monthly").select("month_date", F.round("value", 2).alias("pt_shares_idx"))
    wm = Window.orderBy("month_date")
    m = (m.join(oecd, "month_date", "left")
          .withColumn("ret_brent_eur_m", F.col("brent_eur_avg") / F.lag("brent_eur_avg").over(wm) - 1)
          .withColumn("ret_pt_shares_m", F.col("pt_shares_idx") / F.lag("pt_shares_idx").over(wm) - 1)
          .withColumn("ret_psi_m", F.col("psi_eom") / F.lag("psi_eom").over(wm) - 1)
          .withColumn("yoy_brent_eur", F.col("brent_eur_avg") / F.lag("brent_eur_avg", 12).over(wm) - 1)
          .withColumn("yoy_pt_shares", F.col("pt_shares_idx") / F.lag("pt_shares_idx", 12).over(wm) - 1)
          .withColumn("year", F.year("month_date")))
    # Inflação: IHPC de Portugal → Brent em euros a preços do último mês publicado (para os gráficos de evolução)
    cpi = spark.table("sv_fact_cpi_monthly").select("month_date", F.col("value").alias("hicp_publicado"))
    w_ffm  = Window.orderBy("month_date").rowsBetween(Window.unboundedPreceding, 0)
    w_allm = Window.orderBy("month_date").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
    m = (m.join(cpi, "month_date", "left")
          .withColumn("hicp", F.last("hicp_publicado", ignorenulls=True).over(w_ffm))     # meses ainda sem IHPC: último valor
          .withColumn("deflator", F.last("hicp", ignorenulls=True).over(w_allm) / F.col("hicp"))
          .withColumn("brent_eur_real", F.round(F.col("brent_eur_avg") * F.col("deflator"), 2)))
    m.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_market_monthly")

    # ---- Anual
    # variação anual = valor no fim do ano / valor no fim do ano anterior - 1
    wy = Window.orderBy("year")
    y = (daily.groupBy("year").agg(
            F.round(F.avg("brent_usd"), 2).alias("brent_usd_avg"),
            F.round(F.min("brent_usd"), 2).alias("brent_usd_min"),
            F.round(F.max("brent_usd"), 2).alias("brent_usd_max"),
            F.round(F.avg("brent_eur"), 2).alias("brent_eur_avg"),
            F.round(F.avg("eurusd"), 4).alias("eurusd_avg"),
            F.max_by("brent_eur", "date").alias("brent_eur_eoy"),
            F.max_by("psi_close", "date").alias("psi_eoy"),
            F.round(F.stddev("ret_brent_eur") * math.sqrt(252), 4).alias("vol_brent_eur"),
            F.round(F.stddev("ret_psi") * math.sqrt(252), 4).alias("vol_psi"),
            F.round(F.corr("ret_psi", "ret_brent_eur"), 3).alias("corr_diaria_psi_brent"),
            F.sum(F.col("psi_traded").cast("int")).alias("dias_psi"))
         .join(spark.table("sv_fact_pt_shares_monthly").groupBy(F.year("month_date").alias("year"))
                    .agg(F.max_by("value", "month_date").alias("pt_shares_eoy")), "year", "left")
         .join(spark.table("gd_market_monthly").groupBy("year")
                    .agg(F.round(F.avg("brent_eur_real"), 2).alias("brent_eur_real_avg")), "year", "left")
         .withColumn("var_brent_eur", F.round(F.col("brent_eur_eoy") / F.lag("brent_eur_eoy").over(wy) - 1, 4))
         .withColumn("var_psi", F.round(F.col("psi_eoy") / F.lag("psi_eoy").over(wy) - 1, 4))
         .withColumn("var_pt_shares_oecd", F.round(F.col("pt_shares_eoy") / F.lag("pt_shares_eoy").over(wy) - 1, 4)))
    y.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_market_yearly")

    if TABLE_FORMAT == "delta":
        spark.sql("OPTIMIZE gd_market_daily ZORDER BY (date)")
    for t in ["gd_market_daily", "gd_market_monthly", "gd_market_yearly"]:
        print(f"{t:20} {spark.table(t).count():>6} linhas")

build_gold()

In [ ]:
display(spark.table("gd_market_daily").orderBy(F.desc("date")).limit(5))

<a id="s4-1"></a>
### 4.1 Gold das empresas: retornos mensais e sensibilidade ao petróleo

A hipótese é sobre empresas, por isso tem tabelas próprias: os retornos mensais de cada empresa e uma tabela com o efeito do petróleo em cada uma, já com o veredicto.

Para testar a hipótese empresa a empresa, a análise é **mensal**.

**`gd_equity_monthly`** — uma linha por empresa e por mês:
- retorno mensal logarítmico, com o **fecho ajustado** do último dia de negociação do mês;
- **qualidade:** um dia com variação > 40% no fecho ajustado (provável operação de capital mal ajustada na fonte; ex.: Altri em 2000 e 2005) marca o mês com `ret_outlier`, e esse mês fica fora das estatísticas;
- retorno mensal do Brent em EUR (fim de mês) e o retorno dos 1, 2 e 3 meses anteriores;
- **mercado PT** = média, com pesos iguais, das outras empresas nesse mês (sem a própria empresa);
- cabaz `PT_EX_GALP` = média das 14 empresas sem a Galp;
- correlação móvel de 36 meses com o Brent e preço em euros de hoje (descontando a inflação).

**`gd_oil_sensitivity`** — uma linha por empresa × período (total 2000-2026 e 4 subperíodos):
- correlação mensal e **beta** (`ret_ação ~ ret_brent_eur`): efeito total do petróleo;
- **beta sem mercado** (`ret_ação ~ ret_brent_eur + ret_mercado_pt`): efeito do petróleo depois de descontar o movimento geral da bolsa. Separa o "custo do petróleo" do "ciclo económico", que faz subir e descer petróleo e ações ao mesmo tempo;
- estatística t (significativo se |t| ≥ 2) e **veredicto**: *Confirma* (sinal esperado e significativo), *Contraria* (significativo com o sinal oposto) ou *Sem efeito significativo*;
- correlações com o petróleo de 1, 2 e 3 meses antes (o efeito pode demorar a aparecer).

As regressões são calculadas diretamente com agregações do Spark (variâncias e covariâncias por grupo), por isso o mesmo código serve para 15 ou para milhares de empresas.

In [ ]:
OUTLIER_DAILY = 0.40      # variação diária acima de 40% → provável operação de capital não ajustada na fonte
MIN_MONTHS    = 24        # mínimo de meses para calcular estatísticas
PERIODS = [("Total 2000-2026", 2000, 2026), ("2000-2007", 2000, 2007), ("2008-2013", 2008, 2013),
           ("2014-2019", 2014, 2019), ("2020-2026", 2020, 2026)]
BASKET_ID = "PT_EX_GALP"

def _verdict(beta, t):
    """Confirma = sinal esperado e |t| >= 2; Contraria = |t| >= 2 com o sinal oposto."""
    expected = F.when(F.col("oil_hypothesis") == "+", 1).otherwise(-1)
    return (F.when(F.col("n_meses") < MIN_MONTHS, "Dados insuficientes")
             .when(t.isNull(), None)
             .when(F.abs(t) < 2, "Sem efeito significativo")
             .when(F.signum(beta) == expected, "Confirma")
             .otherwise("Contraria"))

def build_gold_equity():
    dim = (spark.table("sv_dim_instrument").filter(F.col("sector").isNotNull())
                .select("instrument_id", "name", "sector", "oil_hypothesis", "sector_view"))
    eq = spark.table("sv_fact_equity_daily").join(dim.select("instrument_id"), "instrument_id")   # só empresas (sem o PSI)

    # 1) Mensal por empresa: fecho ajustado no último dia de negociação + deteção de saltos diários anómalos
    wd = Window.partitionBy("instrument_id").orderBy("date")
    eq = (eq.withColumn("ret_d", F.col("adj_close") / F.lag("adj_close").over(wd) - 1)
            .withColumn("month_date", F.trunc("date", "month")))
    em = eq.groupBy("instrument_id", "month_date").agg(
        F.max_by("adj_close", "date").alias("adj_close_eom"),
        F.max_by("close", "date").alias("close_eom"),
        F.count("*").alias("dias_negociacao"),
        (F.coalesce(F.max((F.abs("ret_d") > OUTLIER_DAILY).cast("int")), F.lit(0)) == 1).alias("ret_outlier"))
    wm = Window.partitionBy("instrument_id").orderBy("month_date")
    em = (em.withColumn("ret_m", F.when(F.months_between("month_date", F.lag("month_date").over(wm)) == 1,
                                        F.log(F.col("adj_close_eom") / F.lag("adj_close_eom").over(wm))))
            .withColumn("ret_m_stat", F.when(~F.col("ret_outlier"), F.col("ret_m"))))     # retorno usado nas estatísticas

    # 2) Mercado PT: média (pesos iguais) das OUTRAS empresas nesse mês
    mk = em.groupBy("month_date").agg(F.sum("ret_m_stat").alias("_soma"), F.count("ret_m_stat").alias("_n"))
    em = (em.join(mk, "month_date", "left")
            .withColumn("ret_mercado_pt",
                        F.when(F.col("ret_m_stat").isNotNull() & (F.col("_n") > 1),
                               (F.col("_soma") - F.col("ret_m_stat")) / (F.col("_n") - 1))
                         .when(F.col("ret_m_stat").isNull() & (F.col("_n") > 0), F.col("_soma") / F.col("_n")))
            .drop("_soma", "_n"))

    # 3) Cabaz das 14 empresas sem a Galp (tratado como mais um "instrumento")
    basket = (em.filter((F.col("instrument_id") != "GALP.LS") & F.col("ret_m_stat").isNotNull())
                .groupBy("month_date").agg(F.avg("ret_m_stat").alias("ret_m"))
                .withColumn("ret_m_stat", F.col("ret_m"))
                .withColumn("ret_outlier", F.lit(False))
                .withColumn("instrument_id", F.lit(BASKET_ID)))
    em = em.unionByName(basket, allowMissingColumns=True)
    dim_all = dim.unionByName(spark.createDataFrame(
        [(BASKET_ID, "Cabaz 14 empresas (sem Galp)", "Cabaz sem Galp", "-", "-")], dim.schema))

    # 4) Brent em EUR no fim do mês, retorno mensal e desfasamentos
    wb = Window.orderBy("month_date")
    brent = (spark.table("gd_market_daily").withColumn("month_date", F.trunc("date", "month"))
               .groupBy("month_date").agg(F.max_by("brent_eur", "date").alias("brent_eur_eom"))
               .withColumn("ret_brent_eur_m", F.log(F.col("brent_eur_eom") / F.lag("brent_eur_eom").over(wb))))
    for k in (1, 2, 3):
        brent = brent.withColumn(f"ret_brent_eur_m_lag{k}", F.lag("ret_brent_eur_m", k).over(wb))
    defl = spark.table("gd_market_monthly").select("month_date", "deflator")

    w36 = Window.partitionBy("instrument_id").orderBy("month_date").rowsBetween(-35, 0)
    g = (em.join(dim_all, "instrument_id").join(brent, "month_date", "left").join(defl, "month_date", "left")
           .withColumn("adj_close_real", F.round(F.col("adj_close_eom") * F.col("deflator"), 4))
           .withColumn("_ok", (F.col("ret_m_stat").isNotNull() & F.col("ret_brent_eur_m").isNotNull()).cast("int"))
           .withColumn("corr_36m_brent", F.when(F.sum("_ok").over(w36) >= 30,
                                                 F.corr("ret_m_stat", "ret_brent_eur_m").over(w36)))
           .withColumn("year", F.year("month_date"))
           .select("instrument_id", "name", "sector", "oil_hypothesis", "sector_view", "month_date", "year",
                   "dias_negociacao", "close_eom", "adj_close_eom", "adj_close_real", "ret_m", "ret_outlier", "ret_m_stat",
                   "ret_mercado_pt", "brent_eur_eom", "ret_brent_eur_m", "ret_brent_eur_m_lag1", "ret_brent_eur_m_lag2",
                   "ret_brent_eur_m_lag3", "corr_36m_brent"))
    g.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_equity_monthly")

    # 5) Sensibilidade ao petróleo por empresa × período (regressões com agregações do Spark)
    gm = spark.table("gd_equity_monthly").filter(F.col("ret_m_stat").isNotNull() & F.col("ret_brent_eur_m").isNotNull())
    per = spark.createDataFrame(PERIODS, "periodo string, ano_ini int, ano_fim int")
    s = gm.join(F.broadcast(per), (gm.year >= per.ano_ini) & (gm.year <= per.ano_fim))   # cada mês entra no total e no seu subperíodo
    y, x, mk = "ret_m_stat", "ret_brent_eur_m", "ret_mercado_pt"
    agg = s.groupBy("instrument_id", "name", "sector", "oil_hypothesis", "sector_view", "periodo", "ano_ini", "ano_fim").agg(
        F.count("*").alias("n_meses"),
        F.corr(y, x).alias("corr_m"),
        F.var_samp(x).alias("vx"), F.var_samp(mk).alias("vm"), F.var_samp(y).alias("vy"),
        F.covar_samp(x, y).alias("cxy"), F.covar_samp(mk, y).alias("cmy"), F.covar_samp(x, mk).alias("cxm"),
        *[F.corr(y, f"{x}_lag{k}").alias(f"corr_lag{k}") for k in (1, 2, 3)])
    ok = F.col("n_meses") >= MIN_MONTHS
    c = F.col
    res = (agg
        # regressão simples: ret_ação = a + beta·ret_brent
        .withColumn("beta", F.when(ok, c("cxy") / c("vx")))
        .withColumn("se_beta", F.when(ok, F.sqrt((c("vy") - c("beta") * c("cxy")) / c("vx") / (c("n_meses") - 2))))
        .withColumn("t_beta", c("beta") / c("se_beta"))
        # regressão múltipla: ret_ação = a + b1·ret_brent + b2·ret_mercado_pt
        .withColumn("_d", c("vx") * c("vm") - c("cxm") * c("cxm"))
        .withColumn("beta_sem_mercado", F.when(ok, (c("cxy") * c("vm") - c("cmy") * c("cxm")) / c("_d")))
        .withColumn("beta_mercado", F.when(ok, (c("cmy") * c("vx") - c("cxy") * c("cxm")) / c("_d")))
        .withColumn("se_beta_sem_mercado", F.when(ok, F.sqrt(
            (c("vy") - c("beta_sem_mercado") * c("cxy") - c("beta_mercado") * c("cmy")) * c("vm") / c("_d") / (c("n_meses") - 3))))
        .withColumn("t_beta_sem_mercado", c("beta_sem_mercado") / c("se_beta_sem_mercado"))
        .withColumn("veredicto", _verdict(c("beta"), c("t_beta")))
        .withColumn("veredicto_sem_mercado", _verdict(c("beta_sem_mercado"), c("t_beta_sem_mercado")))
        .drop("vx", "vm", "vy", "cxy", "cmy", "cxm", "_d"))
    res.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_oil_sensitivity")
    for t in ["gd_equity_monthly", "gd_oil_sensitivity"]:
        print(f"{t:20} {spark.table(t).count():>6} linhas")

build_gold_equity()

In [ ]:
display(spark.table("gd_oil_sensitivity").filter("periodo = 'Total 2000-2026'")
        .select("name", "sector", "oil_hypothesis", "n_meses", F.round("corr_m", 2).alias("corr_m"),
                F.round("beta", 3).alias("beta"), F.round("t_beta", 1).alias("t"), "veredicto",
                F.round("beta_sem_mercado", 3).alias("beta_sem_mercado"), F.round("t_beta_sem_mercado", 1).alias("t_sem_mercado"),
                "veredicto_sem_mercado")
        .orderBy("sector", "name"))
display(spark.table("gd_equity_monthly").filter("ret_outlier").select("instrument_id", "month_date", "ret_m"))

**Fim da secção 4.** As cinco tabelas Gold estão prontas: três de mercado (diária, mensal e anual) e duas de empresas. A tabela acima já mostra o veredicto de cada empresa, que a secção 6 explica. Antes disso, a secção 5 faz as leituras seguintes da fonte.

---
<a id="s5"></a>
## 5. Leituras incrementais e reprocessamento

Até aqui houve uma só leitura. Esta secção faz as outras duas, cada uma a pedir só o que falta, e mostra que a Silver não fica com duplicados e que se reconstrói a partir da Bronze.

Num processo real o pipeline corre periodicamente. Aqui fazemos mais **duas leituras** da fonte, cada uma a partir da última data já existente na Bronze (*watermark*) com uma **sobreposição de 45 dias**. A sobreposição garante que correções da fonte são apanhadas, mas cria duplicados na Bronze, que a Silver resolve com o `MERGE`.

In [ ]:
OVERLAP_DAYS = 45

def watermarks():
    wf = spark.table("bz_fred_observations").groupBy("series_id").agg(F.max("observation_date").alias("wm"))
    wy = spark.table("bz_yahoo_chart").groupBy("symbol").agg(F.to_date(F.from_unixtime(F.max("ts_epoch"))).cast("string").alias("wm"))
    return {r[0]: r[1] for r in wf.collect() + wy.collect()}

def run_pipeline(batch_name, end_date):
    """Pipeline completo de uma leitura incremental: API → Landing → Bronze → Silver (MERGE) → Gold."""
    wm = watermarks()
    windows = {}
    for s in all_series:
        start = (dt.date.fromisoformat(wm[s]) - dt.timedelta(days=OVERLAP_DAYS)).isoformat() if s in wm else HIST_START
        windows[s] = (start, end_date)
    print(f"=== {batch_name}: leitura até {end_date}")
    bid, files = ingest(batch_name, windows)
    load_bronze(bid, files)
    build_silver(bid)
    build_gold()
    build_gold_equity()
    return bid

batch2_id = run_pipeline("incremental_1", "2026-06-30")

In [ ]:
batch3_id = run_pipeline("incremental_2", dt.date.today().isoformat())

In [ ]:
# Registo de todas as leituras (tabela de controlo do processo)
display(spark.table("ctl_ingestion_log").select("batch_name", "source", "series_id", "window_start", "window_end", "extract_mode", "records", "ingestion_ts")
        .orderBy("ingestion_ts", "series_id"))

In [ ]:
# A Bronze guarda TODAS as leituras (com sobreposições); a Silver tem um único registo por chave
bz_rows = spark.table("bz_fred_observations").groupBy("series_id").agg(
    F.countDistinct("_batch_id").alias("leituras"), F.count("*").alias("linhas_bronze"),
    F.countDistinct("observation_date").alias("datas_distintas"))
sv_rows = (spark.table("sv_fact_price_daily").select("instrument_id").unionAll(spark.table("sv_fact_pt_shares_monthly").select("instrument_id"))
                     .unionAll(spark.table("sv_fact_cpi_monthly").select("instrument_id"))
           .groupBy(F.col("instrument_id").alias("series_id")).agg(F.count("*").alias("linhas_silver")))
display(bz_rows.join(sv_rows, "series_id")
        .withColumn("duplicados_removidos", F.col("linhas_bronze") - F.col("datas_distintas"))
        .withColumn("sem_valor_removidos", F.col("datas_distintas") - F.col("linhas_silver")))

<a id="s5-1"></a>
### 5.1 Reprocessamento a partir da Bronze
Como a Bronze tem todo o histórico de leituras, a Silver pode ser **reconstruída de raiz** a qualquer momento (por exemplo, se uma regra de normalização mudar). O resultado tem de ser idêntico ao obtido incrementalmente.

In [ ]:
before = {t: spark.table(t).count() for t in SILVER_FACTS}
for t in before:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
build_silver(batch_id=None)          # None = todo o histórico da Bronze
after = {t: spark.table(t).count() for t in before}
print("Silver idêntica após reprocessamento:", before == after)

In [ ]:
# Delta Lake: histórico de versões da tabela (time travel) — só no Databricks
if TABLE_FORMAT == "delta":
    display(spark.sql("DESCRIBE HISTORY gd_market_daily").select("version", "timestamp", "operation", "operationMetrics"))
    print("Linhas na versão 0 (após a carga inicial):", spark.sql("SELECT COUNT(*) FROM gd_market_daily VERSION AS OF 0").first()[0])
    print("Linhas na versão atual:", spark.table("gd_market_daily").count())

**Fim da secção 5.** Três leituras: a Bronze guarda as sobreposições (65 linhas repetidas por série diária) e a Silver fica com um registo por chave. A Silver reconstruída de raiz tem as mesmas linhas que a incremental. No Databricks, o `DESCRIBE HISTORY` mostra cada escrita da Gold como uma versão da tabela.

---
<a id="s6"></a>
## 6. Análise

Todas as análises leem só a Gold: primeiro as queries SQL (6.1), depois os gráficos (6.2) e por fim os modelos de ML (6.3). Em cada parte, o mercado vem antes das empresas.
<a id="s6-1"></a>
### 6.1 Queries (Spark SQL sobre a camada Gold)

Cada query responde a uma pergunta concreta, descrita na célula anterior.

**Resumo anual:** preço médio do Brent em USD, em EUR e em EUR a preços de hoje, variação anual do Brent (em EUR), do PSI e do índice OCDE de ações PT, volatilidade e correlação diária PSI–Brent.

In [ ]:
%sql
SELECT year, brent_usd_avg, brent_usd_min, brent_usd_max, brent_eur_avg, brent_eur_real_avg, eurusd_avg,
       ROUND(var_brent_eur*100,1) AS var_brent_eur_pct,
       ROUND(var_pt_shares_oecd*100,1) AS var_acoes_pt_oecd_pct,
       ROUND(var_psi*100,1) AS var_psi_pct,
       ROUND(vol_psi*100,1) AS vol_psi_pct, corr_diaria_psi_brent
FROM gd_market_yearly
ORDER BY year

**Comportamento nas grandes crises:** variação do Brent (EUR) e das ações portuguesas entre o início e o fim de cada episódio (dados mensais, para cobrir 2000–2026).

In [ ]:
%sql
WITH episodios AS (
  SELECT * FROM VALUES
    ('Bolha dot-com',              DATE'2000-03-01', DATE'2003-03-01'),
    ('Subida do petróleo 2007-08', DATE'2007-01-01', DATE'2008-07-01'),
    ('Crise financeira 2008',      DATE'2008-07-01', DATE'2009-03-01'),
    ('Crise da dívida (troika)',   DATE'2010-01-01', DATE'2012-06-01'),
    ('Colapso do petróleo 2014-16',DATE'2014-06-01', DATE'2016-01-01'),
    ('COVID-19',                   DATE'2020-01-01', DATE'2020-04-01'),
    ('Recuperação pós-COVID',      DATE'2020-04-01', DATE'2021-12-01'),
    ('Guerra na Ucrânia',          DATE'2022-01-01', DATE'2022-06-01'),
    ('2026 até hoje',              DATE'2026-01-01', DATE'2026-12-01')
  AS t(episodio, inicio, fim))
SELECT e.episodio, e.inicio, MAX(m.month_date) AS fim_dados,
       ROUND((MAX_BY(m.brent_eur_avg, m.month_date)/MIN_BY(m.brent_eur_avg, m.month_date)-1)*100,1) AS var_brent_eur_pct,
       ROUND((MAX_BY(m.pt_shares_idx, m.month_date) FILTER (WHERE m.pt_shares_idx IS NOT NULL)
             /MIN_BY(m.pt_shares_idx, m.month_date) FILTER (WHERE m.pt_shares_idx IS NOT NULL)-1)*100,1) AS var_acoes_pt_pct,
       ROUND((MAX_BY(m.psi_avg, m.month_date) FILTER (WHERE m.psi_avg IS NOT NULL)
             /MIN_BY(m.psi_avg, m.month_date) FILTER (WHERE m.psi_avg IS NOT NULL)-1)*100,1) AS var_psi_pct
FROM episodios e JOIN gd_market_monthly m ON m.month_date BETWEEN e.inicio AND e.fim
GROUP BY e.episodio, e.inicio ORDER BY e.inicio

**Os 10 dias com maior variação do Brent** (em EUR) desde 2013 e o que o PSI fez nesse mesmo dia.

In [ ]:
%sql
SELECT date, brent_usd, ROUND(ret_brent_eur*100,2) AS brent_eur_pct, psi_close, ROUND(ret_psi*100,2) AS psi_pct
FROM gd_market_daily
WHERE year >= 2013 AND brent_quoted AND psi_traded
ORDER BY ABS(ret_brent_eur) DESC
LIMIT 10

**Regimes do petróleo:** retorno médio mensal das ações portuguesas quando o Brent (EUR) sobe muito, fica estável ou cai muito nesse mês.

In [ ]:
%sql
SELECT CASE WHEN ret_brent_eur_m <= -0.10 THEN '1. Brent cai > 10%'
            WHEN ret_brent_eur_m <  -0.03 THEN '2. Brent cai 3-10%'
            WHEN ret_brent_eur_m <=  0.03 THEN '3. Brent estável (±3%)'
            WHEN ret_brent_eur_m <   0.10 THEN '4. Brent sobe 3-10%'
            ELSE '5. Brent sobe > 10%' END AS regime_petroleo,
       COUNT(*) AS meses,
       ROUND(AVG(ret_pt_shares_m)*100,2) AS ret_medio_acoes_pt_pct,
       ROUND(AVG(CASE WHEN ret_pt_shares_m > 0 THEN 1 ELSE 0 END)*100,0) AS pct_meses_acoes_sobem
FROM gd_market_monthly
WHERE ret_brent_eur_m IS NOT NULL AND ret_pt_shares_m IS NOT NULL
GROUP BY 1 ORDER BY 1

**Correlação** entre retornos do petróleo e das ações portuguesas, por frequência e período.

In [ ]:
dly = spark.table("gd_market_daily").filter("year >= 2013 AND psi_traded")
mth = spark.table("gd_market_monthly")
corr_df = (dly.agg(F.corr("ret_psi", "ret_brent_eur").alias("c")).select(F.lit("Diária 2013–2026: PSI vs Brent EUR").alias("medida"), "c")
    .union(dly.agg(F.corr("ret_psi", "ret_eurusd")).select(F.lit("Diária 2013–2026: PSI vs EUR/USD"), "*"))
    .union(mth.agg(F.corr("ret_pt_shares_m", "ret_brent_eur_m")).select(F.lit("Mensal 2000–2026: ações PT (OCDE) vs Brent EUR"), "*"))
    .union(mth.agg(F.corr("ret_psi_m", "ret_brent_eur_m")).select(F.lit("Mensal 2013–2026: PSI vs Brent EUR"), "*"))
    .union(mth.agg(F.corr("pt_shares_idx", "brent_eur_avg")).select(F.lit("Níveis mensais 2000–2026: ações PT vs Brent EUR"), "*"))
    .withColumn("correlacao", F.round("c", 3)).drop("c"))
display(corr_df)

**Hipótese por empresa (2000–2026, retornos mensais):** efeito total do petróleo (beta) e efeito depois de descontar o movimento da bolsa (beta sem mercado). As colunas `efeito_*_10pct` leem-se como a variação média da ação, em %, num mês em que o Brent (EUR) sobe 10%.

In [ ]:
%sql
SELECT name AS empresa, sector AS setor, oil_hypothesis AS sinal_esperado, n_meses,
       ROUND(corr_m, 2) AS corr_mensal,
       ROUND(beta * 10, 2) AS efeito_total_10pct, ROUND(t_beta, 1) AS t, veredicto,
       ROUND(beta_sem_mercado * 10, 2) AS efeito_sem_mercado_10pct, ROUND(t_beta_sem_mercado, 1) AS t_sem_mercado,
       veredicto_sem_mercado
FROM gd_oil_sensitivity
WHERE periodo = 'Total 2000-2026'
ORDER BY CASE WHEN instrument_id = 'GALP.LS' THEN 0 WHEN instrument_id = 'PT_EX_GALP' THEN 2 ELSE 1 END,
         setor, efeito_sem_mercado_10pct

**Por setor:** média dos efeitos e número de empresas que confirmam ou contrariam a hipótese, antes e depois de descontar a bolsa.

In [ ]:
%sql
SELECT sector AS setor, FIRST(sector_view) AS leitura_setor, COUNT(*) AS empresas,
       ROUND(AVG(corr_m), 2) AS corr_media,
       ROUND(AVG(beta) * 10, 2) AS efeito_total_10pct,
       ROUND(AVG(beta_sem_mercado) * 10, 2) AS efeito_sem_mercado_10pct,
       SUM(CASE WHEN veredicto = 'Confirma' THEN 1 ELSE 0 END) AS confirmam,
       SUM(CASE WHEN veredicto = 'Contraria' THEN 1 ELSE 0 END) AS contrariam,
       SUM(CASE WHEN veredicto_sem_mercado = 'Confirma' THEN 1 ELSE 0 END) AS confirmam_sem_mercado,
       SUM(CASE WHEN veredicto_sem_mercado = 'Contraria' THEN 1 ELSE 0 END) AS contrariam_sem_mercado
FROM gd_oil_sensitivity
WHERE periodo = 'Total 2000-2026' AND instrument_id <> 'PT_EX_GALP'
GROUP BY sector
ORDER BY efeito_sem_mercado_10pct DESC

**Ao longo do tempo:** correlação mensal de cada empresa com o Brent (EUR) por subperíodo (em branco: empresa ainda não cotada ou menos de 24 meses).

In [ ]:
%sql
SELECT name AS empresa, sector AS setor,
       MAX(CASE WHEN periodo = '2000-2007' THEN ROUND(corr_m, 2) END) AS `2000-2007`,
       MAX(CASE WHEN periodo = '2008-2013' THEN ROUND(corr_m, 2) END) AS `2008-2013`,
       MAX(CASE WHEN periodo = '2014-2019' THEN ROUND(corr_m, 2) END) AS `2014-2019`,
       MAX(CASE WHEN periodo = '2020-2026' THEN ROUND(corr_m, 2) END) AS `2020-2026`,
       MAX(CASE WHEN periodo = 'Total 2000-2026' THEN ROUND(corr_m, 2) END) AS total
FROM gd_oil_sensitivity
WHERE n_meses >= 24
GROUP BY name, sector
ORDER BY setor, empresa

**O efeito demora a aparecer?** Correlação do retorno mensal da ação com o retorno do Brent (EUR) no mesmo mês e nos 1, 2 e 3 meses anteriores.

In [ ]:
%sql
SELECT name AS empresa, sector AS setor,
       ROUND(corr_m, 2) AS mesmo_mes, ROUND(corr_lag1, 2) AS `1_mes_antes`,
       ROUND(corr_lag2, 2) AS `2_meses_antes`, ROUND(corr_lag3, 2) AS `3_meses_antes`
FROM gd_oil_sensitivity
WHERE periodo = 'Total 2000-2026'
ORDER BY setor, empresa

**Fim de 6.1.** No mercado, o petróleo e as ações portuguesas movem-se no mesmo sentido, mas pouco: a correlação dos retornos é 0,21 (diária) e 0,25 (mensal). Nas empresas, a Galp é a única com o sinal da hipótese no efeito total. O efeito "custo" só aparece em algumas empresas depois de descontar a bolsa. Os gráficos de 6.2 mostram o mesmo.

<a id="s6-2"></a>
### 6.2 Dashboard

Primeiro o mercado (petróleo e ações portuguesas), depois as empresas. Cada gráfico tem a seguir uma nota de leitura.
Os gráficos abaixo são gerados a partir das tabelas Gold (ficam no HTML exportado).

No Databricks cria-se também um **AI/BI Dashboard** (menu *New → Dashboard*) com estas fontes de dados: `gd_market_daily`, `gd_market_monthly`, `gd_market_yearly`, `gd_equity_monthly` e `gd_oil_sensitivity`. Widgets propostos:
1. Brent mensal 2000–2026 em USD, em EUR e em EUR a preços de hoje (linhas)
2. Correlação móvel de 1 ano PSI–Brent (linha)
3. Efeito do petróleo por empresa, descontando a bolsa (`gd_oil_sensitivity`, barras coloridas por setor)
4. Correlação móvel de 36 meses com o Brent, com filtro por empresa (`gd_equity_monthly`, linhas)
5. Tabela de resumo anual (`gd_market_yearly`)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pm = spark.table("gd_market_monthly").orderBy("month_date").toPandas()
pdl = spark.table("gd_market_daily").filter("year >= 2013").orderBy("date").toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
ax.plot(pm.month_date, pm.brent_usd_avg, label="Brent (USD/barril)", color="#1f4e79")
ax.plot(pm.month_date, pm.brent_eur_avg, label="Brent (EUR/barril)", color="#e07b00")
ax.set_title("Preço do Brent, média mensal 2000–2026"); ax.legend(); ax.grid(alpha=.3)

ax = axes[0, 1]
ax.plot(pm.month_date, pm.pt_shares_idx, color="#2e7d32", label="Ações PT – OCDE (2015=100)")
ax.set_ylabel("Índice ações PT"); ax.grid(alpha=.3)
ax2 = ax.twinx(); ax2.plot(pm.month_date, pm.brent_eur_avg, color="#e07b00", alpha=.6, label="Brent (EUR)")
ax2.set_ylabel("Brent EUR/barril")
ax.set_title("Ações portuguesas vs Brent em EUR (mensal)")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

ax = axes[1, 0]
base = pdl.dropna(subset=["psi_close"]).iloc[0]
ax.plot(pdl.date, pdl.psi_close / base.psi_close * 100, label="PSI", color="#2e7d32")
ax.plot(pdl.date, pdl.brent_eur / base.brent_eur * 100, label="Brent (EUR)", color="#e07b00")
ax.set_title(f"PSI vs Brent EUR, base 100 = {base.date}"); ax.legend(); ax.grid(alpha=.3)

ax = axes[1, 1]
ax.plot(pdl.date, pdl.corr_1y_psi_brent, color="#6a1b9a")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Correlação móvel de 1 ano (252 dias) dos retornos diários PSI vs Brent EUR"); ax.grid(alpha=.3)

plt.tight_layout(); plt.show()

**Como ler os quatro gráficos**
- **Em cima, à esquerda:** Brent em dólares (azul) e em euros (laranja). A distância entre as linhas é o efeito do câmbio: em 2008 a média foi de 97 USD, mas só 65 EUR por barril.
- **Em cima, à direita:** índice OCDE das ações portuguesas (verde, eixo da esquerda) e Brent em euros (laranja, eixo da direita). As escalas são diferentes; o que conta é se as linhas sobem e descem ao mesmo tempo. Nos níveis parecem relacionadas (correlação 0,51), mas isso vem de tendências longas, por isso as conclusões usam retornos.
- **Em baixo, à esquerda:** PSI e Brent em euros com base 100 no primeiro dia de 2013 com cotação do PSI. Mostra o percurso acumulado de cada um, incluindo a subida do Brent em 2026, que o PSI só acompanhou em parte.
- **Em baixo, à direita:** correlação dos retornos diários nos 12 meses anteriores. Acima de zero, PSI e Brent movem-se no mesmo sentido. Foi positiva entre 2014 e 2023, com pico perto de 0,55 em 2016, e ficou ligeiramente negativa em 2024–2026.

A relação existe, é fraca e muda com o tempo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
mm = pm.dropna(subset=["ret_brent_eur_m", "ret_pt_shares_m"])
ax.scatter(mm.ret_brent_eur_m * 100, mm.ret_pt_shares_m * 100, s=12, alpha=.6, color="#1f4e79")
ax.axhline(0, color="grey", lw=.8); ax.axvline(0, color="grey", lw=.8)
ax.set_xlabel("Retorno mensal Brent EUR (%)"); ax.set_ylabel("Retorno mensal ações PT (%)")
ax.set_title("Retornos mensais 2000–2026"); ax.grid(alpha=.3)

ax = axes[1]
py = spark.table("gd_market_yearly").orderBy("year").toPandas()
x = py.year.astype(int)
ax.bar(x - 0.2, py.var_brent_eur * 100, width=.4, label="Brent EUR", color="#e07b00")
ax.bar(x + 0.2, py.var_pt_shares_oecd * 100, width=.4, label="Ações PT (OCDE)", color="#2e7d32")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Variação anual (%)"); ax.legend(); ax.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

**Como ler**
- **À esquerda:** cada ponto é um mês, com o retorno do Brent em euros (horizontal) e o das ações portuguesas (vertical). A nuvem inclina pouco para cima (correlação 0,25). Nos meses em que o Brent cai mais de 10%, as ações caem em média 1,6%.
- **À direita:** variação anual do Brent em euros (laranja) e das ações portuguesas (verde). Barras do mesmo lado do zero são anos em que os dois andaram juntos, como 2008. Em 2010 e 2011, já na crise da dívida, separaram-se: o petróleo subiu e as ações caíram.

O petróleo acompanha o ciclo económico: cai e sobe com as ações nas crises de procura, mas não quando o problema é interno.

#### Gráficos da análise por empresa e da inflação

In [ ]:
# Estilo comum a estes gráficos: paleta validada para daltonismo, eixos e grelha discretos
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

C_BRENT, C_NOMINAL, C_GALP, C_CABAZ = "#eb6834", "#898781", "#2a78d6", "#1baf7a"
SECTOR_COLORS = {"Petróleo e gás": "#2a78d6", "Eletricidade e redes": "#eb6834", "Indústria exportadora": "#1baf7a",
                 "Consumo e retalho": "#eda100", "Serviços domésticos": "#e87ba4",
                 "Exposição a países petrolíferos": "#008300", "Cabaz sem Galp": "#898781"}
SECTOR_ORDER = list(SECTOR_COLORS)
CRISES = [("Crise 2008", "2008-07-01", "2009-03-01"), ("Troika", "2010-01-01", "2012-06-01"),
          ("Colapso do petróleo", "2014-06-01", "2016-01-01"), ("COVID-19", "2020-01-01", "2020-04-01"),
          ("Ucrânia", "2022-01-01", "2022-06-01")]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
                     "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False})

# Brent em euros: nominal vs. a preços de hoje (descontando a inflação portuguesa)
pm = spark.table("gd_market_monthly").orderBy("month_date").toPandas()
pm["month_date"] = pd.to_datetime(pm.month_date)
base_mes = pm.loc[pm.hicp_publicado.notna(), "month_date"].max()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pm.month_date, pm.brent_eur_avg, color=C_NOMINAL, lw=1.5, label="Nominal (euros de cada ano)")
ax.plot(pm.month_date, pm.brent_eur_real, color=C_BRENT, lw=2, label=f"Real (euros de {base_mes:%m/%Y})")
for ano in (2008, 2012, 2022):
    r = pm[pm.month_date.dt.year == ano].sort_values("brent_eur_real").iloc[-1]
    ax.annotate(f"{r.month_date:%m/%Y}: {r.brent_eur_real:.0f} € reais\n({r.brent_eur_avg:.0f} € nominais)",
                (r.month_date, r.brent_eur_real), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=9, color="#52514e")
ax.set_ylabel("EUR por barril (média mensal)")
ax.set_title("Brent em euros: nominal vs. real (descontando a inflação em Portugal)")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

display(spark.table("gd_market_yearly").filter("year IN (2000, 2008, 2012, 2016, 2020, 2022, 2026)")
        .select("year", "brent_usd_avg", "brent_eur_avg", "brent_eur_real_avg").orderBy("year"))

**Como ler**
- Cinzento: preço do barril em euros de cada ano. Laranja: o mesmo preço em euros de hoje, corrigido pela inflação portuguesa (IHPC). As linhas juntam-se no fim porque a base é o último mês com IHPC publicado.
- As anotações marcam o mês mais caro de 2008, 2012 e 2022. Em euros de hoje, a média de 2008 (92 €) fica abaixo das de 2012 (115 €) e de 2022 (111 €).
- A tabela abaixo dá as médias anuais de alguns anos em USD, em EUR e em EUR de hoje.

Para quem paga em euros, o choque de 2008 foi menor do que os de 2012 e 2022.

In [ ]:
# Mapa de calor: correlação móvel de 36 meses (retornos mensais) de cada empresa com o Brent, no fim de cada ano
eqm = spark.table("gd_equity_monthly").toPandas()
eqm["month_date"] = pd.to_datetime(eqm.month_date)
names = (eqm[["instrument_id", "name", "sector"]].drop_duplicates()
         .assign(_s=lambda d: d.sector.map(SECTOR_ORDER.index)).sort_values(["_s", "name"]).reset_index(drop=True))
fim_ano = eqm.sort_values("month_date").groupby(["instrument_id", "year"]).tail(1)   # dezembro (ou último mês disponível)
hm = fim_ano.pivot(index="instrument_id", columns="year", values="corr_36m_brent").reindex(names.instrument_id)
hm = hm.loc[:, hm.notna().any()]

cmap = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
cmap.set_bad("#ffffff")
fig, ax = plt.subplots(figsize=(16, 7.5))
im = ax.imshow(hm.values, cmap=cmap, vmin=-0.6, vmax=0.6, aspect="auto")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names.name)
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels(hm.columns, rotation=90)
ax.grid(False)
for i in range(1, len(names)):                      # separadores e nomes dos setores
    if names.sector[i] != names.sector[i - 1]:
        ax.axhline(i - 0.5, color="#52514e", lw=1)
for sec, grp in names.groupby("sector", sort=False):
    ax.text(len(hm.columns) - 0.3, grp.index.to_series().mean(), sec, va="center", fontsize=9, color="#52514e")
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.22)
cb.set_label("Correlação com o Brent (EUR), últimos 36 meses")
ax.set_title("Correlação móvel de 36 meses entre os retornos mensais de cada empresa e do Brent (fim de cada ano)\n"
             "azul = sobe com o petróleo · vermelho = desce quando o petróleo sobe")
plt.tight_layout(); plt.show()

**Como ler**
- Cada linha é uma empresa, agrupada por setor; cada coluna é o fim de um ano. A cor é a correlação dos retornos mensais com o Brent nos 36 meses anteriores: azul, a ação sobe com o petróleo; vermelho, desce quando o petróleo sobe; branco, sem dados.
- A Galp é azul em quase todo o período. No fim de 2020 a maioria das empresas passa de 0,5: a COVID-19 fez cair e recuperar o petróleo e as ações ao mesmo tempo.
- No fim de 2025 aparece vermelho na EDP (−0,56), na EDP Renováveis (−0,50) e na Jerónimo Martins (−0,41).

O sinal da relação muda com o tipo de choque: as crises de procura juntam petróleo e ações, os choques de oferta separam-nos.

In [ ]:
# Efeito do petróleo por empresa: total vs. descontando o movimento da bolsa (2000-2026)
sens = spark.table("gd_oil_sensitivity").filter("periodo = 'Total 2000-2026'").toPandas()
sens = (sens.assign(_s=sens.sector.map(SECTOR_ORDER.index))
            .sort_values(["_s", "name"], ascending=[False, False]).reset_index(drop=True))
pos = range(len(sens))
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), sharey=True)
for ax, b, se, title in [(axes[0], "beta", "se_beta", "Efeito total do petróleo"),
                         (axes[1], "beta_sem_mercado", "se_beta_sem_mercado", "Efeito descontando o movimento da bolsa")]:
    ok = sens[b].notna()
    ax.barh([p for p, o in zip(pos, ok) if o], sens.loc[ok, b] * 10, xerr=2 * sens.loc[ok, se] * 10, height=0.6,
            color=[SECTOR_COLORS[s] for s in sens.loc[ok, "sector"]],
            error_kw={"ecolor": "#52514e", "elinewidth": 1, "capsize": 2})
    ax.axvline(0, color="#52514e", lw=1)
    ax.set_title(title); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Variação média da ação (%) num mês em que o Brent (EUR) sobe 10%")
axes[0].set_yticks(list(pos)); axes[0].set_yticklabels(sens.name)
fig.legend(handles=[Patch(color=c, label=s) for s, c in SECTOR_COLORS.items()], loc="lower center", ncol=4)
fig.suptitle("Barras de erro = ±2 erros-padrão: se não cruzam o zero, o efeito é estatisticamente significativo (|t| ≥ 2)",
             y=0.995, fontsize=10, color="#52514e")
plt.tight_layout(rect=(0, 0.08, 1, 0.97)); plt.show()

**Como ler**
- Cada barra é a variação média da ação, em %, num mês em que o Brent (EUR) sobe 10%. À esquerda, o efeito total; à direita, o efeito depois de descontar o movimento da bolsa nesse mês. A cor indica o setor.
- A linha preta é o intervalo de ±2 erros-padrão. Se cruza o zero, o efeito não é estatisticamente significativo.
- No efeito total todas as empresas sobem com o petróleo; a Galp sobe +2,7%. Descontando a bolsa, a Galp mantém-se positiva (+2,1%), 7 empresas passam a negativo e só a REN e os CTT têm um efeito negativo significativo.

A hipótese confirma-se na Galp. Nas outras, o efeito "custo" é pequeno e só aparece depois de retirar o ciclo económico.

In [ ]:
# Correlação móvel de 36 meses: Galp vs. cabaz das outras 14 empresas, com as crises assinaladas
fig, ax = plt.subplots(figsize=(14, 5))
for nome, ini, fim in CRISES:
    ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color="#e1e0d9", alpha=0.6, lw=0)
    ax.text(pd.Timestamp(ini), 0.78, nome, fontsize=8, color="#52514e", rotation=90, va="top")
for iid, cor, label in [("GALP.LS", C_GALP, "Galp"), ("PT_EX_GALP", C_CABAZ, "Cabaz sem Galp")]:
    d = eqm[eqm.instrument_id == iid].sort_values("month_date")
    ax.plot(d.month_date, d.corr_36m_brent, color=cor, lw=2, label=label)
    last = d.dropna(subset=["corr_36m_brent"]).iloc[-1]
    ax.annotate(f"{label} {last.corr_36m_brent:.2f}", (last.month_date, last.corr_36m_brent),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color="#0b0b0b")
ax.axhline(0, color="#52514e", lw=1)
ax.set_ylim(-0.4, 0.8)
ax.set_ylabel("Correlação com o Brent (EUR)")
ax.set_title("Correlação móvel de 36 meses dos retornos mensais com o Brent (EUR)")
ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

**Como ler**
- Correlação dos retornos mensais com o Brent (EUR) nos 36 meses anteriores: Galp (azul) e média das outras 14 empresas (verde). As faixas cinzentas são as crises.
- O cabaz sobe nas crises de procura e chega a 0,73 no fim de 2020. Na subida do petróleo de 2026 cai para −0,32, enquanto a Galp fica em 0,50.

Em 2026 a bolsa portuguesa comportou-se como a hipótese prevê: a Galp acompanhou o petróleo e as outras empresas foram em sentido contrário.

In [ ]:
# Base 100 a preços de hoje: Galp, cabaz sem Galp e Brent (EUR), desde a entrada da Galp em bolsa
defl = pm.set_index("month_date")["deflator"]
galp = eqm[eqm.instrument_id == "GALP.LS"].sort_values("month_date").set_index("month_date")
cab = eqm[eqm.instrument_id == "PT_EX_GALP"].sort_values("month_date").set_index("month_date")
inicio = galp.index[1]                                    # primeiro mês completo da Galp em bolsa
idx = pd.DataFrame(index=galp.loc[inicio:].index)
idx["Galp"] = galp.adj_close_real
idx["Brent (EUR)"] = galp.brent_eur_eom * defl.reindex(galp.index)
idx["Cabaz sem Galp"] = np.exp(cab.ret_m.loc[inicio:].iloc[1:].cumsum()).reindex(idx.index).fillna(1.0) * defl.reindex(idx.index)
idx = idx / idx.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 5))
for col, cor in [("Galp", C_GALP), ("Cabaz sem Galp", C_CABAZ), ("Brent (EUR)", C_BRENT)]:
    ax.plot(idx.index, idx[col], color=cor, lw=2, label=col)
    ax.annotate(f"{col} {idx[col].iloc[-1]:.0f}", (idx.index[-1], idx[col].iloc[-1]),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color="#0b0b0b")
ax.axhline(100, color="#52514e", lw=1)
ax.set_ylabel(f"Base 100 = {inicio:%m/%Y} (preços reais)")
ax.set_title(f"Galp, cabaz das outras 14 empresas e Brent (EUR), a preços de {base_mes:%m/%Y} (retornos com dividendos)")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

**Como ler**
- Valor, a preços de hoje, de 100 € aplicados quando a Galp entrou em bolsa (11/2006): na Galp, no cabaz das outras 14 empresas e no Brent em euros. Nas ações, os retornos incluem os dividendos.
- No fim do período a Galp vale 3,9 vezes o valor inicial, o cabaz 2,0 vezes e o Brent 1,3 vezes.

Desde 2006, a Galp rendeu mais do que o próprio petróleo e do que o resto da bolsa portuguesa.

<a id="s6-3"></a>
### 6.3 Modelos de Machine Learning (Spark MLlib)

Duas perguntas diferentes: o petróleo explica o PSI no mesmo dia? E ajuda a prever o dia seguinte? A primeira é uma regressão, a segunda uma classificação.

**Modelo 1: regressão linear (explicativo).** Quanto do movimento diário do PSI é explicado, no mesmo dia, pelo petróleo (Brent EUR) e pelo câmbio EUR/USD? Período 2013–2026.

**Modelo 2: Random Forest (preditivo).** Com a informação disponível até hoje (retornos passados do PSI e do Brent, distância às médias móveis, volatilidade), conseguimos prever se o PSI **sobe amanhã**? Divisão temporal: treino 2013–2023 e teste 2024–2026, sem misturar futuro com passado.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

ml_base = spark.table("gd_market_daily").filter("year >= 2013 AND psi_traded AND ret_psi IS NOT NULL AND ret_brent_eur IS NOT NULL AND ret_eurusd IS NOT NULL")

lr_df = VectorAssembler(inputCols=["ret_brent_eur", "ret_eurusd"], outputCol="features").transform(ml_base)
lr_model = LinearRegression(featuresCol="features", labelCol="ret_psi").fit(lr_df)
print(f"Regressão linear  ret_psi = {lr_model.intercept:.5f} + {lr_model.coefficients[0]:.4f}·ret_brent_eur + {lr_model.coefficients[1]:.4f}·ret_eurusd")
print(f"R² = {lr_model.summary.r2:.3f}  |  RMSE = {lr_model.summary.rootMeanSquaredError:.4f}  |  n = {lr_model.summary.numInstances}")
print(f"Leitura: quando o Brent (EUR) sobe 10%, o PSI sobe em média {lr_model.coefficients[0]*10:.2f}% no mesmo dia.")

In [ ]:
w = Window.orderBy("date")
feat = ml_base
# psi_lag1 = retorno de hoje (conhecido no fecho), psi_lag2 = ontem, ... ; o alvo é o retorno de amanhã
feat = feat.withColumn("psi_lag1", F.col("ret_psi")).withColumn("brent_lag1", F.col("ret_brent_eur"))
for k in range(2, 6):
    feat = feat.withColumn(f"psi_lag{k}", F.lag("ret_psi", k - 1).over(w)).withColumn(f"brent_lag{k}", F.lag("ret_brent_eur", k - 1).over(w))
feat = (feat.withColumn("psi_vs_ma50", F.col("psi_close") / F.col("psi_ma50") - 1)
            .withColumn("psi_vs_ma200", F.col("psi_close") / F.col("psi_ma200") - 1)
            .withColumn("brent_vs_ma50", F.col("brent_eur") / F.col("brent_eur_ma50") - 1)
            .withColumn("label", (F.lead("ret_psi", 1).over(w) > 0).cast("double"))
            .dropna())
features = [f"psi_lag{k}" for k in range(1, 6)] + [f"brent_lag{k}" for k in range(1, 6)] + \
           ["ret_eurusd", "psi_vs_ma50", "psi_vs_ma200", "brent_vs_ma50", "vol_1y_psi", "vol_1y_brent", "corr_1y_psi_brent"]
data = VectorAssembler(inputCols=features, outputCol="features").transform(feat)
train, test = data.filter("year <= 2023"), data.filter("year >= 2024")

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200, maxDepth=5, seed=42).fit(train)
pred = rf.transform(test)
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
baseline = test.agg(F.avg("label")).first()[0]
print(f"Treino: {train.count()} dias | Teste: {test.count()} dias")
print(f"Accuracy = {acc:.3f} | AUC = {auc:.3f} | baseline 'sobe sempre' = {max(baseline, 1-baseline):.3f}")

imp = sorted(zip(features, rf.featureImportances.toArray()), key=lambda x: -x[1])[:8]
display(spark.createDataFrame([(f, float(round(v, 3))) for f, v in imp], "variavel string, importancia double"))

**Fim de 6.3.** O Brent e o câmbio explicam 5% da variação diária do PSI (R²). O Random Forest acerta em 47,5% dos dias (AUC 0,44), abaixo da regra "sobe sempre" (53,4%). O petróleo acompanha a bolsa no próprio dia, mas não ajuda a prever o dia seguinte.

---
<a id="s7"></a>
## 7. Conclusões

Respostas às cinco perguntas da introdução, em quatro partes: os dados (7.1), a hipótese do grupo (7.2), o processo (7.3) e a tecnologia (7.4).

_Os valores citados são da execução de 07-11-2026. Se o notebook voltar a correr mais tarde, os últimos meses podem mudar ligeiramente._

<a id="s7-1"></a>
### 7.1 Sobre os dados

**1. Petróleo: 26 anos de ciclos muito violentos.** O Brent passou de uma média de **28,6 USD** em 2000 para máximos de **144 USD** em julho de 2008. A seguir vieram as quedas para **34 USD** no fim de 2008, para **26 USD** em 2016 (colapso de 2014-16) e para um mínimo de **9 USD** na COVID-19 (abril de 2020). Teve nova subida em 2022 (média de **101 USD**) e voltou a subir em 2026 (média anual de **97 USD** até agora e máximo de **138 USD**; as causas de 2026 ficam fora do âmbito destes dados). **Em euros a preços de hoje** (descontando a inflação portuguesa), a média de 2008 (**92 €**) foi mais baixa do que as de 2012 (**115 €**) e de 2022 (**111 €**): o euro forte amorteceu o choque de 2008. A média de 2000 corresponde a **56 €** de hoje.

**2. O câmbio importa.** Para um investidor europeu, a valorização do euro entre 2002 e 2008 amorteceu a subida: em 2008 a média foi de 97 USD mas só **65 EUR** por barril. Comparar o petróleo com o PSI em euros é a forma correta. Na prática, o EUR/USD quase não se correlaciona com o PSI diário (0,04).

**3. A relação petróleo–bolsa portuguesa existe, mas é fraca e instável.**
- A correlação dos retornos é de **0,21** (diária, 2013–2026) e de **0,25** (mensal, 2000–2026).
- A correlação dos níveis é mais alta (0,51), mas reflete sobretudo tendências comuns de longo prazo, não causalidade.
- A correlação móvel de 1 ano foi positiva entre 2014 e 2023, com pico perto de **0,55 em 2016** (colapso do petróleo, com peso relevante do setor energético no PSI). Depois desceu para zero e ficou **ligeiramente negativa em 2024–2026**: em 2026 o Brent em euros subiu **83%** e o PSI subiu **15%**.

**4. O petróleo funciona como termómetro do ciclo económico.** Nos meses em que o Brent cai mais de 10%, as ações portuguesas recuam em média **–1,6%** e só sobem em 42% dos meses. Nos meses em que o Brent sobe mais de 10%, avançam **+2,4%** e sobem em 72% dos meses. Petróleo e ações tendem a cair juntos nas crises de procura (2008: Brent –60%, ações PT –51%; COVID-19: –70% e –17%) e a recuperar juntos (2020–21).

**5. Há exceções importantes quando o choque é de oferta ou é interno.** Na subida do petróleo de 2007-08 (Brent +104%) as ações PT caíram 20%. Durante a crise da dívida e a troika (2010–12) o Brent subiu 42% e as ações caíram 34%. Nestes episódios o petróleo caro é um custo para a economia, ou os fatores domésticos dominam.

**6. Prever o PSI com o petróleo não funciona.**
- A regressão linear mostra que o Brent e o câmbio explicam só **5% (R²)** da variação diária do PSI: uma subida de 10% do Brent associa-se a +0,8% do PSI no mesmo dia.
- O Random Forest para prever se o PSI sobe no dia seguinte teve **accuracy de 47,5% e AUC de 0,44**. Ficou abaixo da regra ingénua "sobe sempre" (53,4%).
- É o resultado esperado num mercado eficiente: a informação pública do passado não dá vantagem para prever o dia seguinte.

<a id="s7-2"></a>
### 7.2 Sobre a hipótese do grupo (análise mensal por empresa)

**7. A Galp confirma a hipótese.**
- Correlação mensal de **0,42** com o Brent (EUR). Num mês em que o Brent sobe 10%, a Galp sobe em média **+2,7%** (t = 7,1).
- Descontando o movimento da bolsa, o efeito mantém-se: **+2,1%** (t = 5,6).
- A relação é estável em todos os subperíodos (correlação entre 0,43 e 0,52).
- Desde a entrada em bolsa (11/2006), a preços de hoje e com dividendos, a Galp multiplicou o valor por **3,9**, o cabaz das outras 14 empresas por **2,0** e o Brent por **1,3**.

**8. As outras 14 empresas não confirmam a hipótese no efeito total.**
- Todas têm correlação **positiva** com o petróleo (0,01 a 0,27). Nenhuma desce de forma significativa quando o petróleo sobe; 8 sobem de forma significativa (veredicto *Contraria*).
- O cabaz sem a Galp sobe em média **+1,0%** num mês em que o Brent sobe 10% (t = 4,3).
- A razão é o **ciclo económico**: quando a economia mundial cresce, o petróleo e as ações sobem juntos, e o efeito "custo" fica escondido.

**9. Descontando o movimento da bolsa, aparece o efeito "custo" — mas só em alguns setores.**
- 7 das 14 empresas passam a ter efeito negativo: **eletricidade e redes** (EDP, EDP Renováveis, REN), **serviços domésticos** (NOS, CTT) e duas do retalho (Jerónimo Martins, Sonae).
- São significativas a **REN** (−0,4% por cada +10% no Brent, t = −2,4) e os **CTT** (−1,1%, t = −2,4): *Confirma*.
- As **exportadoras** (Navigator, Altri, Semapa, Corticeira) e as empresas com **negócio em países petrolíferos** (BCP, Mota-Engil) continuam positivas, como previa a leitura por setor. A Semapa e a Ibersol são positivas e significativas (*Contraria*).
- O efeito "custo" é pequeno: menos de 1,2% por cada subida de 10% no petróleo.

**10. A relação depende do tipo de choque.**
- Nas **crises de procura** (COVID-19 e a recuperação de 2020–22) petróleo e ações movem-se juntos: no fim de 2020 a correlação de 36 meses do cabaz sem a Galp chegou a **0,73** e a maioria das empresas passou 0,5 (exceções: Jerónimo Martins e CTT, perto de zero).
- Nos **choques de oferta** a relação inverte-se. No fim de 2025 a correlação de 36 meses já era negativa na EDP (−0,56), na EDP Renováveis (−0,50) e na Jerónimo Martins (−0,41). Em 2026, com o Brent (EUR) a subir 83%, todas as empresas exceto a NOS ficaram com correlação negativa ou nula, e o cabaz sem a Galp caiu para **−0,32**. A Galp manteve-se positiva (0,50). É exatamente o padrão da hipótese do grupo.

**11. O efeito não demora a aparecer.** As correlações com o petróleo de 1, 2 e 3 meses antes são praticamente nulas (−0,12 a 0,09): o mercado incorpora o preço do petróleo no próprio mês, o que é coerente com o resultado do modelo de ML.

**Limitações:** só entram as 15 empresas que estão hoje no PSI (enviesamento de sobrevivência); o "mercado PT" é a média com pesos iguais destas empresas; a CTT, a EDP Renováveis e a REN têm histórico mais curto (155 a 232 meses).

<a id="s7-3"></a>
### 7.3 Sobre o processo
- **Medallion de ponta a ponta:** Landing (ficheiros em bruto) → Bronze (histórico *append-only*) → Silver (modelo normalizado) → Gold (tabelas analíticas).
- **Incremental:** foram feitas 3 leituras de 21 séries, 5 do FRED e 16 do Yahoo (carga histórica + 2 incrementais), cada uma a partir do *watermark* com 45 dias de sobreposição.
  - A Bronze acumulou as sobreposições: 65 duplicados por série diária.
  - O `MERGE` da Silver resolveu-os, ficando a versão mais recente de cada registo.
- **Reprocessável:** a Silver foi apagada e reconstruída só a partir da Bronze, com resultado idêntico. É isto que permite corrigir regras de negócio sem voltar à fonte.
- **Qualidade e rastreabilidade:**
  - Foram removidos centenas de dias sem cotação (feriados) e convertidos tipos de texto para número e data.
  - Valores anómalos mas reais (WTI a –37,63 USD em 20-04-2020) ficaram marcados com `dq_flag` em vez de apagados.
  - Nas empresas usou-se o fecho ajustado a dividendos e operações de capital. Dois meses da Altri (2000 e 2005), com saltos diários acima de 40% mal ajustados na fonte, ficaram fora das estatísticas.
  - Cada registo guarda o `_batch_id` e o ficheiro de origem, e a tabela `ctl_ingestion_log` documenta todas as leituras.
- **Limitação de dados:** não há PSI diário gratuito antes de 2013. A solução foi combinar duas fontes: o índice mensal da OCDE para 2000–2026 e o PSI diário para 2013–2026.

<a id="s7-4"></a>
### 7.4 Sobre a tecnologia
- **Apache Spark** permitiu fazer tudo com a mesma API, em Python e SQL:
  - ler CSV e JSON aninhado (`explode`/`posexplode`);
  - funções de janela (forward-fill, retornos, médias móveis, correlação móvel);
  - modelos de ML (MLlib);
  - regressões por empresa e período calculadas só com agregações (variâncias e covariâncias), sem ciclos em Python.
  O volume aqui é pequeno (~7 mil dias e ~97 mil cotações diárias de empresas), mas o mesmo código escala para dados intradiários ou para milhares de ativos sem alterações.
- **Delta Lake** trouxe garantias de base de dados aos ficheiros: transações ACID, `MERGE` para *upserts*, *time travel* (`DESCRIBE HISTORY`, `VERSION AS OF`) e `OPTIMIZE`/`ZORDER` para otimizar a leitura da Gold.
- **Databricks Free Edition** chegou para o projeto (Unity Catalog, Volumes, SQL, dashboards), mas tem limitações:
  - o acesso à internet está restrito, por isso foi criado um mecanismo de contingência com ficheiros de *fallback* que mantém o mesmo formato da API;
  - o compute *serverless* não permite `cache()`;
  - a escala de compute é reduzida.
- **Melhorias futuras:**
  - agendar o pipeline como *Job* diário;
  - usar *Auto Loader* / *Lakeflow Declarative Pipelines* para a ingestão;
  - acrescentar mais variáveis explicativas (taxas de juro, preço do gás natural) e incluir as empresas que saíram da bolsa, para eliminar o enviesamento de sobrevivência;
  - modelar a volatilidade em vez da direção do mercado.